# MLB 2026 Win Prediction — Data Preparation Notes

## 1. Data Sources
- `PlayerTeamsAll_2026.csv` — 2026 projected rosters with roles and weights
- `CareerHittingStatsAlltime.csv` — Career hitting stats for all players
- `CareerFieldingStatsAlltime.csv` — Career fielding stats for all players
- `CareerPitchingStatsAlltime.csv` — Career pitching stats for all players

---

## 2. Historical Team Filtering
Removed all rows belonging to defunct, historical, or Negro League franchises
from the career stats dataframes. This includes teams like the Homestead Grays,
Kansas City Monarchs, Brooklyn Dodgers, and other pre-modern or relocated franchises.
Null team names were also dropped in the same step.

**Affected dataframes:** `career_hitting`, `career_fielding`, `career_pitching`

---

## 3. Team Code Standardization
Career stats use full team names (e.g. "Los Angeles Dodgers") while roster data
uses abbreviations (e.g. "LAD"). A `match()` function was written to convert full
team names to their standard 2-3 letter codes by:

- Building abbreviation candidates from initials (e.g. "LAD", "LA", "L")
- Handling edge cases explicitly:
  - Arizona Diamondbacks → `ARI`
  - Washington Nationals / Montreal Expos → `WSN`
  - Florida Marlins → `MIA`
  - Chicago White Sox → `CHW`
  - All Angels franchises → `LAA`
  - All Tampa Bay franchises → `TBR`
  - St. Louis Cardinals → `STL`

Rows that could not be matched were tagged `"No Match"` for review.

**Result:** Team code column added to `career_hitting`, `career_fielding`, `career_pitching`

---

## 4. Column Renaming — Inference to Training Alignment
Raw API column names were mapped to match the training data schema using
rename dictionaries:

- **Offense:** e.g. `atBats → AB`, `homeRuns → HR`, `baseOnBalls → BB`
- **Pitching:** e.g. `P_wins → W_pitch`, `P_era → ERA_pitch`, `P_inningsPitched → IP_pitch`

---

## 5. Derived Columns
Columns present in inference data but missing from training data were computed:

### Hitting
| Column | Formula |
|--------|---------|
| `totalBases` | Singles + 2×2B + 3×3B + 4×HR |
| `stolenBasePercentage` | SB / (SB + CS) |
| `caughtStealingPercentage` | CS / (SB + CS) |
| `atBatsPerHomeRun` | AB / HR |
| `plateAppearances` | AB + BB (approximate) |
| `babip` | (H - HR) / (AB - SO - HR) (approximate, missing SF) |

### Pitching
| Column | Formula |
|--------|---------|
| `P_strikeoutWalkRatio` | SO / BB |
| `P_winPercentage` | W / (W + L) |
| `P_strikeoutsPer9Inn` | (SO / IP) × 9 |
| `P_walksPer9Inn` | (BB / IP) × 9 |
| `P_hitsPer9Inn` | (H / IP) × 9 |
| `P_homeRunsPer9` | (HR / IP) × 9 |
| `P_runsScoredPer9` | (R / IP) × 9 |

---

## 6. Team-Level Projection — Hitting (`aggregateHitting`)
Player-level stats aggregated to team-level projections using a weighted average 
approach. `Player_Weight` is derived from FanGraphs projected plate appearances:
`Player_Weight = Proj_PA / sum(Proj_PA)` within each team, so players with more 
projected playing time contribute proportionally more to the team aggregate.

---

## 7. Team-Level Projection — Pitching (`aggregatePitching`)
Same weighted approach as hitting. `Player_Weight` for pitchers is derived from 
FanGraphs projected innings: `Player_Weight = Proj_IP / sum(Proj_IP)` within each team.

**Minimum innings threshold:** Rate columns (ERA, WHIP, AVG_pitch, and all per-9 stats) 
are nulled out for any pitcher with fewer than **30 career IP**. Pitchers below this 
threshold have unreliable rate stats driven by tiny samples (e.g. a 0.00 ERA from 2 
innings, or a 27.00 ERA from a single bad outing). Their tally contributions are still 
counted; only their rate stats are excluded. Weights are renormalized within each team 
over the remaining pitchers with valid data so the weighted average remains properly 
calibrated.

**Rate columns:** ERA, WHIP, AVG_pitch and all per-9 / ratio stats

### Save Column Weighting
Saves and save opportunities projected using role-based weights:
- If both Closer and Setup Men present: **70% Closer / 30% Setup Men**
- If only Closers: 100% split evenly among closers
- If only Setup Men: 100% split evenly among setup men

---

## 8. Mean Shifting
After scaling, all tally projections were shifted so the league-wide mean
matches the historical training data mean. This corrects for systematic
over/under-estimation in the projection methodology without distorting
the relative spread between teams.
```python
shift = historical_mean - projected_mean
projected_column = projected_column + shift
```

In [727]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')
data_dir = os.path.join(os.getcwd(), '..', 'data')
scripts_dir = os.path.join(os.getcwd(), '..', 'scripts')


## **Team Filtering & Code Standardization**
Loaded career hitting, fielding, and pitching stats and removed all defunct/historic
franchises (Negro League, relocated, and pre-modern teams) as well as null team names.
Applied a `match()` function to convert full team names to standard abbreviations,
with explicit handling for edge cases like the Diamondbacks, Angels, and Tampa Bay franchises.
Rows with no matching code are tagged `"No Match"` for review.

In [729]:
historic_teams = [
    "Homestead Grays",
    "Kansas City Athletics",
    "Brooklyn Superbas",
    "Washington Senators",
    "Jacksonville Red Caps",
    "Newark Eagles",
    "Chicago American Giants",
    "St. Louis Stars",
    "Brooklyn Dodgers",
    "St. Louis Browns",
    "Indianapolis ABC's",
    "Memphis Red Sox",
    "New York Black Yankees",
    "Birmingham Black Barons",
    "Kansas City Monarchs",
    "Louisville Black Caps",
    "Oakland Athletics",
    "New York Cubans",
    "Washington Elite Giants",
    "Brooklyn Robins",
    "New York Giants",
    "Washington Pilots",
    "New York Highlanders",
    "Chicago Giants",
    "Indianapolis Athletics",
    "Nashville Elite Giants",
    "Akron Grays",
    "Montgomery Grey Sox",
    "Little Rock Grays",
    "Indianapolis Clowns",
    "Toledo Tigers",
    "Toledo-Indianapolis Crawfords",
    "Louisville White Sox",
    "St. Louis-New Orleans Stars",
    "Cuban Stars West",
    "Hilldale Daisies",
    "Chicago Orphans",
    "Brooklyn Royal Giants",
    "St. Louis Giants",
    "New York Lincoln Giants",
    "Harrisburg Giants",
    "Monroe Monarchs",
    "Washington Black Senators",
    "Brooklyn Eagles",
    "Cuban Stars East",
    "Newark Dodgers",
    "Pollock's Cuban Stars",
    "Dayton Marcos",
    "Newark Browns",
    "Toledo Crawfords",
    "Harrisburgh-St. Louis Stars",
    "Todelo Tigers",
     'Detroit Wolves',
    'Cleveland Red Sox'
]
non_current_teams = [
    "Philadelphia Stars",
    "Cleveland Indians",
    "Detroit Stars",
    "Boston Braves",
    "Philadelphia Bacharach Giants",
    "Cleveland Bears",
    "California Angels",
    "Philadelphia Athletics",
    "Houston Colt 45's",
    "Baltimore Elite Giants",
    "Montreal Expos",
    "Cleveland Buckeyes",
    "Anaheim Angels",
    "Cincinnati-Indianapolis Clowns",
    "Cleveland Blues",
    "Boston Beaneaters",
    "Cleveland Bronchos",
    "Tampa Bay Devil Rays",
    "Boston Doves",
    "Cincinnati Redlegs",
    "Florida Marlins",
    "Boston Bees",
    "Atlanta Black Crackers",
    "Cleveland Naps",
    "Milwaukee Braves",
    "Atlantic City Bacharach Giants",
    "Baltimore Black Sox",
    "Pittsburgh Keystones",
    "Pittsburgh Crawfords",
    "Cole's American Giants",
    "Cincinnati Clowns",
    "Cincinnati Buckeyes",
    "Milwaukee Bears",
    "Boston Americans",
    "Cincinnati Tigers",
    "Cleveland Tate Stars",
    "Cincinnati Cuban Stars",
    "Columbus Buckeyes",
    "Boston Rustlers",
    "Baltimore Sox",
    "Columbus Elite Giants",
    "Columbus Blue Birds",
    "Cleveland Stars",
    "Seattle Pilots"
]
historic_teams+=non_current_teams
rosters=pd.read_csv(os.path.join(data_dir,'PlayerTeamsAll_2026.csv'))
rosters['first_name']=rosters['Name'].apply(lambda x: x.split()[0])
rosters['last_name']=rosters['Name'].apply(lambda x: " ".join(x.split()[1:]))
career_hitting=pd.read_csv(os.path.join(data_dir,'CareerHittingStatsAlltime.csv'))
career_fielding=pd.read_csv(os.path.join(data_dir,'CareerFieldingStatsAlltime.csv'))
career_pitching=pd.read_csv(os.path.join(data_dir,'CareerPitchingStatsAlltime.csv'))
rosters

# 1. Filter out historic teams AND remove null values in one step
career_hitting = career_hitting[
    (~career_hitting['team_name'].isin(historic_teams)) & (career_hitting['team_name'].notna())
]

career_fielding = career_fielding[
    (~career_fielding['team_name'].isin(historic_teams)) & (career_fielding['team_name'].notna())
]

career_pitching = career_pitching[
    (~career_pitching['team_name'].isin(historic_teams)) & (career_pitching['team_name'].notna())
]

def match(s1,codes):
    L=[]
    one=s1.split(' ')
    id=''
    for item in one:
        id+=item[0].upper()
        L.append(id)
    L.append(one[0][0:3].upper())
    if s1=='Arizona Diamondbacks':
        return 'ARI'
    elif s1 in ['Washington Nationals','Montreal Expos']:
        return 'WSN'
    elif s1=='St. Louis Cardinals':
        return 'STL'
    elif s1=='Chicago Cubs':
        return 'CHC'
    elif s1=='Toronto Blue Jays':
        return 'TOR'
    elif s1=='Florida Marlins':
        return 'MIA'
    elif 'Chicago White Sox' in s1:
        return 'CHW'
    elif "Angels" in s1:
        return "LAA"
    elif "Tampa Bay" in s1:
        return "TBR"

    for code in codes:
        for item in L:
            if item==code:
                return code

    return (s1,'No Match')

# 1. Ensure your 'codes' list contains the 30 active abbreviations
# (Based on our previous list: LAD, PHI, BAL, CIN, MIL, etc.)
codes = rosters['Team'].unique()

# 2. Apply the match function to create the 'Team' column in all dataframes
# We use a lambda to ensure we get a string back even on 'No Match'
for df in [career_hitting, career_fielding, career_pitching]:
    df['Team'] = df['team_name'].apply(lambda x: match(x, codes))
    
    # Cleaning up the 'No Match' tuple if it occurs
    df['Team'] = df['Team'].apply(lambda x: x if isinstance(x, str) else "No Match")

# 3. Quick verification check
print(f"Unique Teams in Hitting: {career_hitting['Team'].nunique()}")
print(career_hitting[['team_name', 'Team']].head())

for name, df in zip(['Hitting', 'Fielding', 'Pitching'], [career_hitting, career_fielding, career_pitching]):
    no_match_count = (df['Team'] == "No Match").sum()
    total_rows = len(df)
    percent = (no_match_count / total_rows) * 100
    print(f"{name} - No Match: {no_match_count} ({percent:.1f}%)")

roster_codes = set(rosters['Team'].unique())

for name, df in zip(['Hitting', 'Fielding', 'Pitching'], [career_hitting, career_fielding, career_pitching]):
    no_match = (df['Team'] == 'No Match').sum()
    unmatched_codes = set(df[df['Team'] != 'No Match']['Team'].unique()) - roster_codes
    missing_from_career = roster_codes - set(df['Team'].unique())
    
    print(f"\n── {name} ──")
    print(f"  Total rows:            {len(df)}")
    print(f"  No Match:              {no_match} ({no_match/len(df)*100:.1f}%)")
    print(f"  Codes not in rosters:  {unmatched_codes if unmatched_codes else '✓ None'}")
    print(f"  Roster teams missing:  {missing_from_career if missing_from_career else '✓ All 30 present'}")


Unique Teams in Hitting: 30
                team_name Team
3     Los Angeles Dodgers  LAD
9   Philadelphia Phillies  PHI
12      Baltimore Orioles  BAL
13        Cincinnati Reds  CIN
14  Philadelphia Phillies  PHI
Hitting - No Match: 0 (0.0%)
Fielding - No Match: 0 (0.0%)
Pitching - No Match: 0 (0.0%)

── Hitting ──
  Total rows:            13338
  No Match:              0 (0.0%)
  Codes not in rosters:  ✓ None
  Roster teams missing:  {nan}

── Fielding ──
  Total rows:            31317
  No Match:              0 (0.0%)
  Codes not in rosters:  ✓ None
  Roster teams missing:  {nan}

── Pitching ──
  Total rows:            7768
  No Match:              0 (0.0%)
  Codes not in rosters:  ✓ None
  Roster teams missing:  {nan}


## Two-Way Player & Position Filtering
Ensure players are assigned to the correct career stat datasets based on position.
Pitchers are removed from the hitting set, and the pitching set is restricted to
pitchers and two-way players only. Ohtani is used as a spot-check to confirm
two-way players appear correctly in both datasets.

In [731]:
from IPython.display import display

# ── Hitting ──
print("=== HITTING ===")
print("Unique Position Abbreviations:", career_hitting['position_abbr'].unique())
print("Unique Position Names:        ", career_hitting['position_name'].unique())
display(career_hitting[career_hitting['first_name'] == 'Shohei'])

career_hitting = career_hitting[career_hitting['position_name'] != 'Pitcher']
print(f"Rows remaining after removing Pitchers: {len(career_hitting)}\n")

# ── Pitching ──
print("=== PITCHING ===")
print("Unique Position Abbreviations:", career_pitching['position_abbr'].unique())
print("Unique Position Names:        ", career_pitching['position_name'].unique())
display(career_pitching[career_pitching['first_name'] == 'Shohei'])

career_pitching = career_pitching[career_pitching['position_name'].isin(['Pitcher', 'Two-Way Player'])]
print(f"Rows remaining after filtering for Pitcher/Two-Way: {len(career_pitching)}")

=== HITTING ===
Unique Position Abbreviations: ['P' 'PH' 'C' 'RF' 'X' '3B' 'LF' '2B' 'SS' '1B' 'CF' 'DH' 'PR']
Unique Position Names:         ['Pitcher' 'Pinch Hitter' 'Catcher' 'Outfielder' 'Unknown' 'Third Base'
 'Second Base' 'Shortstop' 'First Base' 'Designated Hitter' 'Pinch Runner']


,player_id,full_name,first_name,last_name,position_code,position_name,position_abbr,team_id,team_name,league_id,...,groundIntoDoublePlay,sacFlies,babip,groundOuts,airOuts,numberOfPitches,leftOnBase,groundOutsToAirouts,catchersInterference,Team
2400,660271,Shohei Ohtani,Shohei,Ohtani,10,Designated Hitter,DH,119.0,Los Angeles Dodgers,104.0,...,49.0,20.0,.325,800.0,796.0,17041.0,1420.0,1.01,13.0,LAD


Rows remaining after removing Pitchers: 7194

=== PITCHING ===
Unique Position Abbreviations: ['CF' 'SS' '1B' 'P' 'C' 'DH' '2B' '3B' 'RF' 'LF' 'TWP' 'OF']
Unique Position Names:         ['Outfielder' 'Shortstop' 'First Base' 'Pitcher' 'Catcher'
 'Designated Hitter' 'Second Base' 'Third Base' 'Two-Way Player'
 'Outfield']


,player_id,full_name,first_name,last_name,position_code,position_name,position_abbr,team_id,team_name,league_id,...,walksPer9Inn,hitsPer9Inn,runsScoredPer9,homeRunsPer9,inheritedRunners,inheritedRunnersScored,catchersInterference,sacBunts,sacFlies,Team
1325,660271,Shohei Ohtani,Shohei,Ohtani,Y,Two-Way Player,TWP,119.0,Los Angeles Dodgers,104.0,...,3.1,6.61,3.13,0.95,0.0,0.0,2.0,2.0,12.0,LAD


Rows remaining after filtering for Pitcher/Two-Way: 7238


## Restricted List Players — Team Assignment & Removal
Manually assign teams for players with null Team values, then drop all restricted
list players since they are ineligible to play and carry null projections.

In [733]:
# All four are on restricted lists — manually assign their teams
team_map = {
    'Luis Ortiz':        'CLE',   # Cleveland Guardians — on restricted list, pitch-rigging case
    'Emmanuel Clase':    'CLE',   # Cleveland Guardians — on restricted list, pitch-rigging case
    'Michael Lorenzen':  'COL',   # Colorado Rockies — signed Jan 2026
    'Wander Franco':     'TBR',   # Tampa Bay Rays — on restricted list, legal issues in DR
}

for name, team in team_map.items():
    rosters.loc[rosters['Name'] == name, 'Team'] = team
print("Restricted players being dropped:")
print(rosters[rosters['Status'] == 'RESTRICTED'][['Name', 'Team', 'Pos', 'ProjectedOpeningDayRole']])

rosters = rosters[rosters['Status'] != 'RESTRICTED']
print(f"\nRoster size after dropping restricted players: {len(rosters)}")

print('Null Team Count in Rosters: ', rosters['Team'].isnull().sum())

Restricted players being dropped:
                  Name Team Pos ProjectedOpeningDayRole
204   Jurickson Profar  ATL  OF         Restricted List
591         Luis Ortiz  CLE  SP         Restricted List
592     Emmanuel Clase  CLE  RP         Restricted List
1839     Wander Franco  TBR  SS         Restricted List

Roster size after dropping restricted players: 2057
Null Team Count in Rosters:  0


In [734]:
rosters[rosters['Team']=='ATH']
print(f"Rosters unique teams:          {rosters['Team'].nunique()}")
print(f"Career hitting unique teams:   {career_hitting['team_name'].nunique()}")
print(f"Career fielding unique teams:  {career_fielding['team_name'].nunique()}")
print(f"Career pitching unique teams:  {career_pitching['team_name'].nunique()}")

# Also check what AL West teams are in rosters
print("\nAL West teams in rosters:")
print(rosters[rosters['Team'].isin(['HOU','SEA','TEX','LAA','OAK','ATH'])]['Team'].unique())
print('Fielding')
print(career_fielding['team_name'].unique())
print(sorted(career_fielding['Team'].unique()))
print()
print(career_hitting['team_name'].unique())
print(sorted(career_hitting['Team'].unique()))
print()

print(sorted(rosters['Team'].unique()))

Rosters unique teams:          30
Career hitting unique teams:   30
Career fielding unique teams:  30
Career pitching unique teams:  30

AL West teams in rosters:
['ATH' 'HOU' 'LAA' 'SEA' 'TEX']
Fielding
['St. Louis Cardinals' 'Boston Red Sox' 'Pittsburgh Pirates'
 'San Francisco Giants' 'Los Angeles Dodgers' 'Philadelphia Phillies'
 'Kansas City Royals' 'Chicago White Sox' 'Cincinnati Reds'
 'Detroit Tigers' 'New York Mets' 'Chicago Cubs' 'Houston Astros'
 'Baltimore Orioles' 'New York Yankees' 'San Diego Padres' 'Texas Rangers'
 'Minnesota Twins' 'Los Angeles Angels' 'Milwaukee Brewers'
 'Seattle Mariners' 'Arizona Diamondbacks' 'Toronto Blue Jays'
 'Atlanta Braves' 'Colorado Rockies' 'Tampa Bay Rays' 'Miami Marlins'
 'Washington Nationals' 'Athletics' 'Cleveland Guardians']
['ARI', 'ATH', 'ATL', 'BAL', 'BOS', 'CHC', 'CHW', 'CIN', 'CLE', 'COL', 'DET', 'HOU', 'KCR', 'LAA', 'LAD', 'MIA', 'MIL', 'MIN', 'NYM', 'NYY', 'PHI', 'PIT', 'SDP', 'SEA', 'SFG', 'STL', 'TBR', 'TEX', 'TOR', 'WSN']



## **Duplicate Name Audit**
Identifies active roster players whose names map to multiple `player_id` entries
in the historical data — indicating a name conflict between two different players.
These cases require manual vetting to ensure career stats are merged onto the
correct player.

In [736]:
def get_at_risk_names(roster_df, history_df, roster_name_col='Name', history_name_col='full_name'):
    """
    Identifies names in the active roster that have multiple 
    player_id entries in the historical dataset.
    """
    # 1. Identify all names in history associated with more than one player_id
    historical_id_counts = history_df.groupby(history_name_col)['player_id'].nunique()
    historical_duplicate_names = historical_id_counts[historical_id_counts > 1].index
    
    # 2. Find which roster names are in that historical duplicate list
    at_risk_mask = roster_df[roster_name_col].isin(historical_duplicate_names)
    at_risk_names = roster_df.loc[at_risk_mask, roster_name_col].unique().tolist()
    
    # Printing summary for visibility
    print(f"--- Conflict Audit ---")
    print(f"Total historical names with multiple IDs: {len(historical_duplicate_names)}")
    print(f"Active roster names flagged for manual vetting: {len(at_risk_names)}")
    
    return at_risk_names
all_positions = ['SS', '2B', 'RF', '3B', 'CF', 'C', '1B', 'DH', 'LF', 'INF/OF',
                 'SP', 'OF', 'RP', 'SP/RP', 'INF', '1B/3B', '3B/1B', '1B/OF',
                 'C/1B', 'OF/1B', 'OF/INF', 'C/INF', '1B/C', 'SS/2B', 'UTL',
                 'RP/SP', 'SS/2B/3B', 'CF/RF/LF', 'C/OF', '1B/LF', '2B/3B/1B']

pitching_positions = [pos for pos in all_positions if 'P' in pos]
print(f"Pitching positions: {pitching_positions}")
pitching_rosters=rosters[rosters['Pos'].isin(pitching_positions)]
hitting_rosters=rosters[~rosters['Pos'].isin(pitching_positions)]

# --- Apply to Hitting ---
print("\nHITTING AUDIT:")
hitting_at_risk = get_at_risk_names(hitting_rosters, career_hitting)
print(hitting_at_risk)
# --- Apply to Pitching ---
print("\nPITCHING AUDIT:")
pitching_at_risk = get_at_risk_names(pitching_rosters, career_pitching)
print(pitching_at_risk)
# --- Apply to Fielding ---
# Note: Since fielding usually covers both pitchers and hitters, 
# you can check it against the full rosters.
print("\nFIELDING AUDIT:")
fielding_at_risk = get_at_risk_names(rosters, career_fielding)
print(fielding_at_risk)


Pitching positions: ['SP', 'RP', 'SP/RP', 'RP/SP']

HITTING AUDIT:
--- Conflict Audit ---
Total historical names with multiple IDs: 64
Active roster names flagged for manual vetting: 5
['Jacob Wilson', 'Max Muncy', 'Nick Allen', 'Josh Bell', 'Luis Matos']

PITCHING AUDIT:
--- Conflict Audit ---
Total historical names with multiple IDs: 66
Active roster names flagged for manual vetting: 6
['Eduardo Rodriguez', 'Juan Morillo', 'Danny Young', 'Javy Guerra', 'Logan Allen', 'Carlos Hernández']

FIELDING AUDIT:
--- Conflict Audit ---
Total historical names with multiple IDs: 196
Active roster names flagged for manual vetting: 13
['Eduardo Rodriguez', 'Ryan Thompson', 'Max Muncy', 'Danny Young', 'Javy Guerra', 'José Ramírez', 'Carlos Hernández', 'Nick Allen', 'Will Smith', 'Eury Pérez', 'Josh Bell', 'Luis Matos', 'Jose Herrera']


## **Roster Splitting, Duplicate Removal & Career Stat Merging**
Positions are split into pitching (`SP`, `RP`, `SP/RP`, `RP/SP`) and hitting sets.
Flagged duplicate names are isolated and removed from both rosters and career stat
dataframes before merging. Hitting and fielding stats are collapsed and joined onto
the hitting roster, while pitching stats are merged onto the pitching roster.
Pitching columns are prefixed with `P_` to avoid conflicts during later combination.

In [738]:
# ── Position Splitting ──────────────────────────────────────────────────────
all_positions = ['SS', '2B', 'RF', '3B', 'CF', 'C', '1B', 'DH', 'LF', 'INF/OF',
                 'SP', 'OF', 'RP', 'SP/RP', 'INF', '1B/3B', '3B/1B', '1B/OF',
                 'C/1B', 'OF/1B', 'OF/INF', 'C/INF', '1B/C', 'SS/2B', 'UTL',
                 'RP/SP', 'SS/2B/3B', 'CF/RF/LF', 'C/OF', '1B/LF', '2B/3B/1B']

pitching_positions = [pos for pos in all_positions if 'P' in pos]
print(f"Pitching positions: {pitching_positions}")

pitching_rosters = rosters[rosters['Pos'].isin(pitching_positions)]
hitting_rosters  = rosters[~rosters['Pos'].isin(pitching_positions)]
print(f"Hitting roster size:  {len(hitting_rosters)}")
print(f"Pitching roster size: {len(pitching_rosters)}")

# ── Remove Duplicate-Name Players from Rosters and Career Stats ─────────────
# Isolate flagged rows for inspection
hitting_rosters_dup_names  = hitting_rosters[hitting_rosters['Name'].isin(hitting_at_risk)]
pitching_rosters_dup_names = pitching_rosters[pitching_rosters['Name'].isin(pitching_at_risk)]

# Clean rosters — remove ambiguous names
clean_hitting_rosters  = hitting_rosters[~hitting_rosters['Name'].isin(hitting_at_risk)]
clean_pitching_rosters = pitching_rosters[~pitching_rosters['Name'].isin(pitching_at_risk)]

# Clean career stats — remove ambiguous names from all three datasets
all_risky_names        = hitting_at_risk + pitching_at_risk
clean_career_hitting   = career_hitting[~career_hitting['full_name'].isin(all_risky_names)]
clean_career_fielding  = career_fielding[~career_fielding['full_name'].isin(all_risky_names)]
clean_career_pitching  = career_pitching[~career_pitching['full_name'].isin(all_risky_names)]

print(f"\nPlayers removed due to name conflicts: {len(all_risky_names)}")
print(f"  Hitting:  {len(hitting_rosters)} → {len(clean_hitting_rosters)} roster rows")
print(f"  Pitching: {len(pitching_rosters)} → {len(clean_pitching_rosters)} roster rows")

# ── Merge Hitting + Career Hitting ──────────────────────────────────────────
merged_hitting = pd.merge(clean_hitting_rosters, clean_career_hitting.drop(columns=['Team']),
                          left_on='Name', right_on='full_name', how='inner')
print(f"\nHitting merge: {len(clean_hitting_rosters)} roster → {len(merged_hitting)} after inner join")

# ── Collapse Fielding Stats (one row per player) ─────────────────────────────
# Fielding can have multiple rows per player (one per position played)
# We collapse to one row per player using gamesPlayed as a tiebreaker
fielding_columns = [
    'first_name', 'last_name', 'full_name', 'player_id', 'gamesPlayed',
    'gamesStarted', 'assists', 'putOuts', 'errors', 'chances', 'fielding',
    'rangeFactorPerGame', 'rangeFactorPer9Inn', 'innings', 'games',
    'doublePlays', 'triplePlays', 'throwingErrors', 'passedBall',
    'caughtStealing', 'stolenBases', 'stolenBasePercentage',
    'caughtStealingPercentage', 'catcherERA', 'catchersInterference',
    'wildPitches', 'pickoffs', 'position'
]

# ── Collapse Fielding Stats (one row per player) ─────────────────────────────
# Exclude DH, P, and aggregate rows (no position) — fielding stats are meaningless for these
# Sum errors across positions, take first for everything else
exclude_cols = ['position_code', 'position_name', 'position_abbr', 'position'] + duplicate_columns

fielding_for_collapse = clean_career_fielding[
    clean_career_fielding['position_abbr'].notna() &
    ~clean_career_fielding['position_abbr'].isin(['DH', 'P', 'TWP'])
]

agg_dict = {
    col: 'sum' if col == 'errors' else 'first'
    for col in fielding_columns
    if col not in exclude_cols + ['first_name', 'last_name', 'full_name', 'player_id']
}

fielding_collapsed = (
    fielding_for_collapse
    .groupby(['first_name', 'last_name', 'full_name', 'player_id'])
    .agg(agg_dict)
    .reset_index()
)

fielding_columns_reduced = [col for col in fielding_columns if col not in exclude_cols]
fielding_reduced = fielding_collapsed[fielding_columns_reduced]

# ── Merge Offense (Hitting + Fielding) ───────────────────────────────────────
merged_offense = pd.merge(merged_hitting, fielding_reduced, on='full_name', how='inner')
print(f"Offense merge (hitting + fielding): {len(merged_offense)} rows")

# Verify Luke Raley
print(merged_offense[merged_offense['full_name'] == 'Luke Raley'][['full_name', 'errors', 'fielding']])

# ── Merge Pitching + Career Pitching ────────────────────────────────────────
merged_pitching = pd.merge(pitching_rosters, clean_career_pitching.drop(columns=['Team']),
                           left_on='Name', right_on='full_name', how='inner')
print(f"Pitching merge: {len(pitching_rosters)} roster → {len(merged_pitching)} after inner join")

# ── Prefix Pitching Columns with P_ ─────────────────────────────────────────
# Roster columns keep their names; all stat columns get a P_ prefix
# to avoid conflicts when offense and pitching are later combined
def reframe_columns(col):
    roster_columns = [
        'Team', 'Team_x', 'Name', 'Pos', 'Status', 'ProjectedOpeningDayRole',
        'Age', 'ServiceTime', 'Is40Man', 'Options', 'Proj_PA', 'Proj_IP',
        'Proj_PT', 'first_name', 'last_name'
    ]
    if col in roster_columns:
        return 'Team' if col in ['Team_x', 'P_Team_x', 'P_P_Team_x'] else col
    return f"P_{col}"

merged_pitching.columns = [reframe_columns(col) for col in merged_pitching.columns]

# ── Verification ─────────────────────────────────────────────────────────────
print(f"\n── Final Shapes ──")
print(f"  merged_offense:  {merged_offense.shape}")
print(f"  merged_pitching: {merged_pitching.shape}")
print(f"\n── Duplicate name rows set aside for manual review ──")
print(f"  Hitting:  {len(hitting_rosters_dup_names)} players")
print(f"  Pitching: {len(pitching_rosters_dup_names)} players")
display(hitting_rosters_dup_names[['Name', 'Pos', 'Team']]) if len(hitting_rosters_dup_names) > 0 else print("  None")
display(pitching_rosters_dup_names[['Name', 'Pos', 'Team']]) if len(pitching_rosters_dup_names) > 0 else print("  None")

Pitching positions: ['SP', 'RP', 'SP/RP', 'RP/SP']
Hitting roster size:  961
Pitching roster size: 1096

Players removed due to name conflicts: 11
  Hitting:  961 → 955 roster rows
  Pitching: 1096 → 1090 roster rows

Hitting merge: 955 roster → 653 after inner join
Offense merge (hitting + fielding): 594 rows
      full_name  errors  fielding
470  Luke Raley     6.0     0.994
Pitching merge: 1096 roster → 809 after inner join

── Final Shapes ──
  merged_offense:  (594, 79)
  merged_pitching: (809, 87)

── Duplicate name rows set aside for manual review ──
  Hitting:  6 players
  Pitching: 6 players


,Name,Pos,Team
79,Jacob Wilson,SS,ATH
84,Max Muncy,3B,ATH
758,Nick Allen,INF,HOU
956,Max Muncy,3B,LAD
1157,Josh Bell,1B,MIN
1668,Luis Matos,OF,SFG


,Name,Pos,Team
14,Eduardo Rodriguez,SP,ARI
43,Juan Morillo,RP,ARI
178,Danny Young,RP,ATL
212,Javy Guerra,RP,ATL
559,Logan Allen,SP,CLE
585,Carlos Hernández,RP,CLE


## Manual Resolution of Duplicate Hitting Names
Manually inspect and resolve players whose names matched multiple `player_id` entries.
Incorrect historical player records are dropped by `player_id`, and name conflicts
(e.g. two players named Max Muncy) are disambiguated by renaming one entry before merging.
Resolved duplicate rows are then concatenated back into the main `merged_offense_clean` dataset.

In [740]:
hitting_names=hitting_rosters_dup_names['Name'].values

for name in hitting_names:
    display(career_hitting[career_hitting['full_name']==name].sort_values(by='full_name'))
    print()
career_stats_dups_hitting=career_hitting[career_hitting['full_name'].isin(hitting_names)].sort_values(by='full_name').reset_index(drop=True)
print('Career Stats Hitting')
display(career_stats_dups_hitting)
idx_to_drop=[458679,607111,275928,110139]
# Create a new dataframe without those IDs
career_stats_dups_hitting = career_stats_dups_hitting[~career_stats_dups_hitting['player_id'].isin(idx_to_drop)]
print('Career Stats After Drop')
career_stats_dups_hitting.loc[
    career_stats_dups_hitting['player_id'] == 571970, 'full_name'
] = 'Max Muncy Dodger'


display(career_stats_dups_hitting)

hitting_rosters_dups=hitting_rosters[hitting_rosters['Name'].isin(hitting_names)].sort_values(by='Name').reset_index(drop=True)
hitting_rosters_dups.loc[
    (hitting_rosters_dups['Name'] == 'Max Muncy') & (hitting_rosters_dups['Team'] == 'LAD'), 'Name'
] = 'Max Muncy Dodger'
merged_hitting_dups=pd.merge(hitting_rosters_dups,career_stats_dups_hitting.drop(columns=['Team']), left_on='Name', right_on='full_name')
print('Joined with Rosters')
display(merged_hitting_dups)

career_stats_dups_fielding=career_fielding[career_fielding['full_name'].isin(hitting_names)].sort_values(by='full_name').reset_index(drop=True)
print('Career Stats fielding')
display(career_stats_dups_fielding)

# Apply same fielding collapse logic as main pipeline
career_stats_dups_fielding = (
    career_stats_dups_fielding[
        career_stats_dups_fielding['position_abbr'].notna() &
        ~career_stats_dups_fielding['position_abbr'].isin(['DH', 'P', 'TWP'])
    ]
    .groupby(['first_name', 'last_name', 'full_name', 'player_id'])
    .agg(agg_dict)
    .reset_index()
)

muncy_dodger_id = 571970  # whatever his actual player_id is

career_stats_dups_fielding.loc[
    career_stats_dups_fielding['player_id'] == muncy_dodger_id, 'full_name'
] = 'Max Muncy Dodger'
career_stats_dups_fielding = career_stats_dups_fielding[~career_stats_dups_fielding['player_id'].isin(idx_to_drop)]
print('Fielding after drop')
display(career_stats_dups_fielding)


career_stats_dups_fielding=career_stats_dups_fielding[fielding_columns_reduced]
merged_offense_dups=pd.merge(merged_hitting_dups,career_stats_dups_fielding, on='full_name')
merged_offense_dups

merged_offense_clean=pd.concat([merged_offense,merged_offense_dups])

pitching_rosters_dup_names
print('all code executed')

,player_id,full_name,first_name,last_name,position_code,position_name,position_abbr,team_id,team_name,league_id,...,groundIntoDoublePlay,sacFlies,babip,groundOuts,airOuts,numberOfPitches,leftOnBase,groundOutsToAirouts,catchersInterference,Team
1406,805779,Jacob Wilson,Jacob,Wilson,6,Shortstop,SS,133.0,Athletics,103.0,...,20.0,2.0,.311,216.0,144.0,2084.0,211.0,1.50,0.0,ATH
14150,607111,Jacob Wilson,Jacob,Wilson,5,Third Base,3B,117.0,Houston Astros,103.0,...,1.0,0.0,.176,9.0,5.0,85.0,9.0,1.80,0.0,HOU


,player_id,full_name,first_name,last_name,position_code,position_name,position_abbr,team_id,team_name,league_id,...,groundIntoDoublePlay,sacFlies,babip,groundOuts,airOuts,numberOfPitches,leftOnBase,groundOutsToAirouts,catchersInterference,Team
8675,571970,Max Muncy,Max,Muncy,5,Third Base,3B,119.0,Los Angeles Dodgers,104.0,...,36.0,33.0,.252,689.0,913.0,17004.0,1547.0,0.75,0.0,LAD
10155,691777,Max Muncy,Max,Muncy,5,Third Base,3B,133.0,Athletics,103.0,...,1.0,1.0,.269,51.0,44.0,824.0,79.0,1.16,0.0,ATH


,player_id,full_name,first_name,last_name,position_code,position_name,position_abbr,team_id,team_name,league_id,...,groundIntoDoublePlay,sacFlies,babip,groundOuts,airOuts,numberOfPitches,leftOnBase,groundOutsToAirouts,catchersInterference,Team
8363,110139,Nick Allen,Nick,Allen,2,Catcher,C,113.0,Cincinnati Reds,104.0,...,NaN,NaN,.272,NaN,NaN,NaN,NaN,NaN,NaN,CIN
10184,669397,Nick Allen,Nick,Allen,6,Shortstop,SS,144.0,Atlanta Braves,104.0,...,21.0,6.0,.263,345.0,291.0,4260.0,413.0,1.19,0.0,ATL


,player_id,full_name,first_name,last_name,position_code,position_name,position_abbr,team_id,team_name,league_id,...,groundIntoDoublePlay,sacFlies,babip,groundOuts,airOuts,numberOfPitches,leftOnBase,groundOutsToAirouts,catchersInterference,Team
8675,571970,Max Muncy,Max,Muncy,5,Third Base,3B,119.0,Los Angeles Dodgers,104.0,...,36.0,33.0,.252,689.0,913.0,17004.0,1547.0,0.75,0.0,LAD
10155,691777,Max Muncy,Max,Muncy,5,Third Base,3B,133.0,Athletics,103.0,...,1.0,1.0,.269,51.0,44.0,824.0,79.0,1.16,0.0,ATH


,player_id,full_name,first_name,last_name,position_code,position_name,position_abbr,team_id,team_name,league_id,...,groundIntoDoublePlay,sacFlies,babip,groundOuts,airOuts,numberOfPitches,leftOnBase,groundOutsToAirouts,catchersInterference,Team
5106,605137,Josh Bell,Josh,Bell,3,First Base,1B,120.0,Washington Nationals,104.0,...,139.0,41.0,.283,1381.0,1048.0,20288.0,2039.0,1.32,10.0,WSN
11632,458679,Josh Bell,Josh,Bell,5,Third Base,3B,110.0,Baltimore Orioles,103.0,...,8.0,0.0,.278,74.0,53.0,1058.0,131.0,1.40,0.0,BAL


...

Career Stats fielding


,player_id,full_name,first_name,last_name,position_code,position_name,position_abbr,team_id,team_name,league_id,...,caughtStealing,stolenBases,stolenBasePercentage,caughtStealingPercentage,catcherERA,catchersInterference,wildPitches,pickoffs,position,Team
0,805779,Jacob Wilson,Jacob,Wilson,6,Shortstop,SS,133.0,Athletics,103.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'code': '6', 'name': 'Shortstop', 'type': 'In...",ATH
1,805779,Jacob Wilson,Jacob,Wilson,10,Designated Hitter,DH,133.0,Athletics,103.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'code': '6', 'name': 'Shortstop', 'type': 'In...",ATH
2,605137,Josh Bell,Josh,Bell,10,Designated Hitter,DH,120.0,Washington Nationals,104.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'code': '3', 'name': 'First Base', 'type': 'I...",WSN
3,605137,Josh Bell,Josh,Bell,3,First Base,1B,120.0,Washington Nationals,104.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'code': '3', 'name': 'First Base', 'type': 'I...",WSN
4,458679,Josh Bell,Josh,Bell,10,Designated Hitter,DH,110.0,Baltimore Orioles,103.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAL
5,458679,Josh Bell,Josh,Bell,9,Outfielder,RF,110.0,Baltimore Orioles,103.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAL
6,458679,Josh Bell,Josh,Bell,7,Outfielder,LF,110.0,Baltimore Orioles,103.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAL
7,458679,Josh Bell,Josh,Bell,5,Third Base,3B,110.0,Baltimore Orioles,103.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAL
8,605137,Josh Bell,Josh,Bell,9,Outfielder,RF,120.0,Washington Nationals,104.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'code': '3', 'name': 'First Base', 'type': 'I...",WSN
9,275928,Luis Matos,Luis,Matos,8,Outfielder,CF,110.0,Baltimore Orioles,103.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAL


Fielding after drop


,first_name,last_name,full_name,player_id,gamesStarted,assists,putOuts,errors,chances,fielding,...,rangeFactorPer9Inn,innings,games,doublePlays,triplePlays,throwingErrors,passedBall,catcherERA,wildPitches,pickoffs
0,Jacob,Wilson,Jacob Wilson,805779,151.0,308.0,171.0,8.0,487.0,0.984,...,3.34,1289.2,151,62.0,0.0,2.0,NaN,None,NaN,NaN
2,Josh,Bell,Josh Bell,605137,884.0,564.0,6514.0,128.0,7142.0,0.991,...,8.45,7542.2,910,597.0,1.0,27.0,NaN,None,NaN,NaN
4,Luis,Matos,Luis Matos,682641,18.0,1.0,29.0,6.0,32.0,0.938,...,1.66,163.0,31,1.0,0.0,2.0,NaN,None,NaN,NaN
5,Max,Muncy,Max Muncy Dodger,571970,249.0,123.0,1958.0,42.0,2095.0,0.993,...,8.23,2275.0,332,143.0,0.0,5.0,NaN,None,NaN,NaN
6,Max,Muncy,Max Muncy,691777,21.0,42.0,34.0,8.0,80.0,0.950,...,3.70,185.0,23,8.0,0.0,2.0,NaN,None,NaN,NaN
8,Nick,Allen,Nick Allen,669397,38.0,97.0,76.0,10.0,178.0,0.972,...,4.58,340.1,51,22.0,0.0,3.0,NaN,None,NaN,NaN


all code executed


## Manual Resolution of Duplicate Pitching Names
Inspect pitchers whose names matched multiple historical records and resolve by
selecting the entry with the highest `player_id` (most recent player). Resolved
rows are prefixed with `P_` and concatenated back into `merged_pitching_clean`.

In [742]:
# ── Inspect duplicate pitching names ────────────────────────────────────────
print('Current Roster Pitchers with Historical Duplicates')
display(pitching_rosters_dup_names.sort_values(by='Name'))

# Pull all career pitching rows for flagged names
career_pitching_dups = career_pitching[career_pitching['full_name'].isin(all_risky_names)]
print('All Career Pitching Rows for Flagged Names')
display(career_pitching_dups.sort_values(by='full_name'))

# ── Resolve duplicates by keeping the highest player_id (most recent) ────────
career_pitching_dups_reduced = career_pitching_dups.groupby('full_name').apply(
    lambda x: x.loc[x['player_id'].idxmax()]
).reset_index(drop=True)
print('After Removing Inactive Players (highest player_id kept)')
display(career_pitching_dups_reduced.sort_values(by='full_name'))

# ── Merge resolved duplicates onto their roster rows ────────────────────────
merged_pitching_dups = pd.merge(
    pitching_rosters_dup_names, career_pitching_dups_reduced.drop(columns=['Team']),
    left_on='Name', right_on='full_name'
)
print('Duplicate Pitchers Merged with Roster')
display(merged_pitching_dups)

# ── Apply P_ prefix and concatenate back into main pitching dataset ──────────
merged_pitching_dups.columns = [reframe_columns(col) for col in merged_pitching_dups.columns]
merged_pitching_clean = pd.concat([merged_pitching, merged_pitching_dups])
print(f'merged_pitching_clean: {merged_pitching_clean.shape}')

Current Roster Pitchers with Historical Duplicates


,Team,Name,Pos,Status,ProjectedOpeningDayRole,Age,ServiceTime,Is40Man,Options,Proj_PA,Proj_IP,Proj_PT,first_name,last_name
585,CLE,Carlos Hernández,RP,NRI,Bullpen Candidate,29.0,4.075,N,0,NaN,18.0,18.0,Carlos,Hernández
178,ATL,Danny Young,RP,Inj,Projected Injured List,31.8,1.160,Y,0,NaN,20.0,20.0,Danny,Young
14,ARI,Eduardo Rodriguez,SP,ACTIVE,Starting Rotation,32.9,10.070,Y,NaN,NaN,132.0,132.0,Eduardo,Rodriguez
212,ATL,Javy Guerra,RP,NRI,Reassigned,30.5,3.003,N,0,NaN,NaN,NaN,Javy,Guerra
43,ARI,Juan Morillo,RP,MiLB (40),Bullpen Candidate,27.0,0.111,Y,2,NaN,30.0,30.0,Juan,Morillo
559,CLE,Logan Allen,SP,ACTIVE,Starting Rotation,27.5,2.093,Y,1,NaN,121.0,121.0,Logan,Allen


All Career Pitching Rows for Flagged Names


,player_id,full_name,first_name,last_name,position_code,position_name,position_abbr,team_id,team_name,league_id,...,walksPer9Inn,hitsPer9Inn,runsScoredPer9,homeRunsPer9,inheritedRunners,inheritedRunnersScored,catchersInterference,sacBunts,sacFlies,Team
6192,400056,Carlos Hernández,Carlos,Hernández,1,Pitcher,P,117.0,Houston Astros,104.0,...,4.8,9.12,4.69,1.21,1.0,0.0,0.0,7.0,3.0,HOU
7715,672578,Carlos Hernández,Carlos,Hernández,1,Pitcher,P,114.0,Cleveland Guardians,103.0,...,4.41,8.89,5.47,0.99,67.0,25.0,0.0,2.0,15.0,CLE
4199,664849,Danny Young,Danny,Young,1,Pitcher,P,121.0,New York Mets,104.0,...,3.71,8.31,4.6,0.59,32.0,7.0,0.0,3.0,0.0,NYM
11623,277415,Danny Young,Danny,Young,1,Pitcher,P,112.0,Chicago Cubs,104.0,...,18.00,15.00,21.00,3.00,0.0,0.0,0.0,0.0,0.0,CHC
3775,121350,Eduardo Rodriguez,Eduardo,Rodriguez,1,Pitcher,P,158.0,Milwaukee Brewers,103.0,...,3.96,8.35,4.27,0.8,242.0,77.0,1.0,43.0,30.0,MIL
4900,593958,Eduardo Rodriguez,Eduardo,Rodriguez,1,Pitcher,P,109.0,Arizona Diamondbacks,104.0,...,3.14,8.83,4.42,1.15,2.0,0.0,2.0,19.0,34.0,ARI
4107,457915,Javy Guerra,Javy,Guerra,1,Pitcher,P,119.0,Los Angeles Dodgers,104.0,...,3.6,9.13,4.28,0.86,127.0,53.0,1.0,14.0,11.0,LAD
9482,642770,Javy Guerra,Javy,Guerra,1,Pitcher,P,139.0,Tampa Bay Rays,103.0,...,6.14,9.86,6.71,1.29,18.0,7.0,0.0,1.0,3.0,TBR
4924,666661,Juan Morillo,Juan,Morillo,1,Pitcher,P,109.0,Arizona Diamondbacks,104.0,...,5.24,9.96,4.98,0.52,22.0,10.0,0.0,2.0,1.0,ARI
11287,434623,Juan Morillo,Juan,Morillo,1,Pitcher,P,115.0,Colorado Rockies,104.0,...,5.91,12.66,13.50,4.22,0.0,0.0,0.0,1.0,0.0,COL


After Removing Inactive Players (highest player_id kept)


,player_id,full_name,first_name,last_name,position_code,position_name,position_abbr,team_id,team_name,league_id,...,walksPer9Inn,hitsPer9Inn,runsScoredPer9,homeRunsPer9,inheritedRunners,inheritedRunnersScored,catchersInterference,sacBunts,sacFlies,Team
0,672578,Carlos Hernández,Carlos,Hernández,1,Pitcher,P,114.0,Cleveland Guardians,103.0,...,4.41,8.89,5.47,0.99,67.0,25.0,0.0,2.0,15.0,CLE
1,664849,Danny Young,Danny,Young,1,Pitcher,P,121.0,New York Mets,104.0,...,3.71,8.31,4.6,0.59,32.0,7.0,0.0,3.0,0.0,NYM
2,593958,Eduardo Rodriguez,Eduardo,Rodriguez,1,Pitcher,P,109.0,Arizona Diamondbacks,104.0,...,3.14,8.83,4.42,1.15,2.0,0.0,2.0,19.0,34.0,ARI
3,642770,Javy Guerra,Javy,Guerra,1,Pitcher,P,139.0,Tampa Bay Rays,103.0,...,6.14,9.86,6.71,1.29,18.0,7.0,0.0,1.0,3.0,TBR
4,666661,Juan Morillo,Juan,Morillo,1,Pitcher,P,109.0,Arizona Diamondbacks,104.0,...,5.24,9.96,4.98,0.52,22.0,10.0,0.0,2.0,1.0,ARI
5,671106,Logan Allen,Logan,Allen,1,Pitcher,P,114.0,Cleveland Guardians,103.0,...,3.58,9.42,4.79,1.33,0.0,0.0,2.0,5.0,10.0,CLE


Duplicate Pitchers Merged with Roster


,Team,Name,Pos,Status,ProjectedOpeningDayRole,Age,ServiceTime,Is40Man,Options,Proj_PA,...,strikeoutsPer9Inn,walksPer9Inn,hitsPer9Inn,runsScoredPer9,homeRunsPer9,inheritedRunners,inheritedRunnersScored,catchersInterference,sacBunts,sacFlies
0,ARI,Eduardo Rodriguez,SP,ACTIVE,Starting Rotation,32.9,10.070,Y,NaN,NaN,...,8.95,3.14,8.83,4.42,1.15,2.0,0.0,2.0,19.0,34.0
1,ARI,Juan Morillo,RP,MiLB (40),Bullpen Candidate,27.0,0.111,Y,2,NaN,...,9.44,5.24,9.96,4.98,0.52,22.0,10.0,0.0,2.0,1.0
2,ATL,Danny Young,RP,Inj,Projected Injured List,31.8,1.160,Y,0,NaN,...,11.57,3.71,8.31,4.6,0.59,32.0,7.0,0.0,3.0,0.0
3,ATL,Javy Guerra,RP,NRI,Reassigned,30.5,3.003,N,0,NaN,...,6.43,6.14,9.86,6.71,1.29,18.0,7.0,0.0,1.0,3.0
4,CLE,Logan Allen,SP,ACTIVE,Starting Rotation,27.5,2.093,Y,1,NaN,...,7.59,3.58,9.42,4.79,1.33,0.0,0.0,2.0,5.0,10.0
5,CLE,Carlos Hernández,RP,NRI,Bullpen Candidate,29.0,4.075,N,0,NaN,...,7.96,4.41,8.89,5.47,0.99,67.0,25.0,0.0,2.0,15.0


merged_pitching_clean: (815, 87)


## **Column Cleanup — Drop `_y` Duplicates and Strip `_x` Suffixes**
Remove duplicate `_y` columns introduced by merges and clean up residual `_x` suffixes.

In [744]:
import re
merged_offense_clean=merged_offense_clean[[col for col in merged_offense_clean.columns if '_y' not in col]]
merged_pitching_clean=merged_pitching_clean[[col for col in merged_pitching_clean.columns if '_y' not in col]]
for col in merged_offense_clean.columns:
    if 'x' in col:
        print(col)
print()
for col in merged_pitching_clean.columns:
    if 'x' in col:
        print(col)
merged_offense_clean.columns=[ re.sub('_x',"",col) for col in merged_offense_clean.columns]
merged_pitching_clean.columns=[ re.sub('_x',"",col) for col in merged_pitching_clean.columns]

first_name_x
last_name_x
player_id_x

P_first_name_x
P_last_name_x


## Null Handling & Player Weight Calculation
Catcher-specific stats (`passedBall`, `wildPitches`, `pickoffs`, `catcherERA`) are filled
with zero for non-catchers. Projection columns (`Proj_PA`, `Proj_IP`, `Proj_PT`) are
zero-filled for players without projections. Player weights are then calculated as each
player's share of their team's total projected plate appearances (offense) or innings
pitched (pitching), so weights sum to 1.0 per team.

In [746]:
print('Length of Merged Offense: ', len(merged_offense_clean))
for key, value in  merged_offense_clean.isnull().sum().items():
    print(f"{key}: {value}")

catcher_columns=["passedBall",
"catcherERA",
"wildPitches",
"pickoffs"]
catcher=merged_offense_clean[catcher_columns+['Name','position_name']]


# .any(axis=1) collapses the 4 columns into a single True/False Series
catcher_filtered = catcher[catcher[catcher_columns].notnull().any(axis=1)]
display(catcher_filtered)
# Only these columns get filled; others stay null
merged_offense_clean.fillna({
    "passedBall": 0,
    "wildPitches": 0,
    "pickoffs": 0,
    "catcherERA": 0  # Or maybe fill with the league average?
}, inplace=True)
print('Catcher/First Basemen Stats filled with zero if null')
print()
for key, value in  merged_offense_clean.isnull().sum().items():
    print(f"{key}: {value}")
# Filter for rows where Proj_PA is not null
null_plate_appearances = merged_offense_clean[merged_offense_clean['Proj_PA'].isnull()]

# Optional: Sort by projected plate appearances to see your heavy hitters at the top
null_plate_appearances  = null_plate_appearances .sort_values(by='Proj_PA', ascending=False)
display(null_plate_appearances [['Name','Team','Proj_PA','ServiceTime','Is40Man','Status']])

# Apply zero-fill to the Offense projections
offense_proj_cols = ['Proj_PT', 'Proj_PA', 'Proj_IP']

merged_offense_clean[offense_proj_cols] = merged_offense_clean[offense_proj_cols].fillna(0)

# Verify the fix for Offense
print("Offense Nulls Remaining:")
print(merged_offense_clean[offense_proj_cols].isnull().sum())




# Calculate total projected PA per team
team_pa_totals = merged_offense_clean.groupby('Team')['Proj_PA'].sum()

# Weight = Player PA / Team Total PA
# We use .get() or a mapping to ensure the 'Team' index matches correctly
merged_offense_clean['Player_Weight'] = merged_offense_clean.apply(
    lambda x: x['Proj_PA'] / team_pa_totals[x['Team']] if team_pa_totals[x['Team']] > 0 else 0, 
    axis=1
)
merged_pitching_clean['Proj_IP'] = merged_pitching_clean['Proj_IP'].fillna(0)
# 1. Calculate the team total IP and broadcast it to every row
team_ip_totals = merged_pitching_clean.groupby('Team')['Proj_IP'].transform('sum')

# 2. Calculate the weight (Player IP / Team Total IP)
# We use a small epsilon (1e-9) to prevent division by zero for teams with no IP
merged_pitching_clean['P_Player_Weight'] = merged_pitching_clean['Proj_IP'] / (team_ip_totals + 1e-9)

# 3. Clean up any cases where both were 0 (result would be 0.0)
merged_pitching_clean['P_Player_Weight'] = merged_pitching_clean['P_Player_Weight'].fillna(0)

# Apply the fill to both cleaned dataframes
merged_offense_clean['ServiceTime'] = merged_offense_clean['ServiceTime'].fillna(0)
merged_pitching_clean['ServiceTime'] = merged_pitching_clean['ServiceTime'].fillna(0)

# Verify: Sum of weights for the 'LAD' or 'NYY' should be approx 1.0
print('Verfify Sums add to 1')
print('Pitching')
print(merged_pitching_clean.groupby('Team')['P_Player_Weight'].sum().head())
print('Offense')
print(merged_offense_clean.groupby('Team')['Player_Weight'].sum().head())

def check_weights(df, weight_col, label):
    # Calculate sums per team
    team_sums = df.groupby('Team')[weight_col].sum()
    
    # Define a tiny buffer (epsilon)
    epsilon = 1e-6
    
    # Find teams that are NOT approximately 1.0
    # Note: We exclude teams with 0.0 sum (likely teams with no projections)
    invalid_teams = team_sums[(team_sums > 0) & ((team_sums < 1 - epsilon) | (team_sums > 1 + epsilon))]
    
    print(f"--- {label} Weight Check ---")
    if invalid_teams.empty:
        print(f"✅ All {label} team weights sum to 1.0 (within buffer).")
    else:
        print(f"❌ Warning! The following teams have invalid sums:")
        print(invalid_teams)



# Run the checks
check_weights(merged_offense_clean, 'Player_Weight', 'Offense')
check_weights(merged_pitching_clean, 'P_Player_Weight', 'Pitching')

Length of Merged Offense:  600
Team: 0
Name: 0
Pos: 0
Status: 0
ProjectedOpeningDayRole: 0
Age: 0
ServiceTime: 2
Is40Man: 0
Options: 181
Proj_PA: 82
Proj_IP: 599
Proj_PT: 82
first_name: 0
last_name: 0
player_id: 0
full_name: 0
position_code: 0
position_name: 0
position_abbr: 0
team_id: 0
team_name: 0
league_id: 0
league: 0
num_teams: 0
gamesPlayed: 0
runs: 0
doubles: 0
triples: 0
homeRuns: 0
baseOnBalls: 0
hits: 0
hitByPitch: 0
avg: 0
atBats: 0
obp: 0
slg: 0
ops: 0
stolenBases: 0
plateAppearances: 0
totalBases: 0
rbi: 0
sacBunts: 0
atBatsPerHomeRun: 0
strikeOuts: 0
intentionalWalks: 0
caughtStealing: 0
stolenBasePercentage: 0
caughtStealingPercentage: 0
groundIntoDoublePlay: 0
sacFlies: 0
babip: 0
groundOuts: 0
airOuts: 0
numberOfPitches: 0
leftOnBase: 0
groundOutsToAirouts: 0
catchersInterference: 0
first_name: 0
last_name: 0
gamesStarted: 0
assists: 0
putOuts: 0
errors: 0
chances: 0
fielding: 0
rangeFactorPerGame: 0
rangeFactorPer9Inn: 0
innings: 0
games: 0
doublePlays: 0
triplePlays

,passedBall,catcherERA,wildPitches,pickoffs,Name,position_name
5,7.0,5.79,66.0,2.0,Gabriel Moreno,Catcher
9,38.0,4.14,290.0,10.0,James McCann,Catcher
15,0.0,5.66,7.0,1.0,Adrian Del Castillo,Catcher
16,4.0,6.5,40.0,1.0,Aramis Garcia,Catcher
21,2.0,5.71,7.0,0.0,Tyler Soderstrom,First Base
...,...,...,...,...,...,...
569,4.0,2.31,39.0,3.0,Tyler Heineman,Catcher
580,18.0,4.82,149.0,12.0,Keibert Ruiz,Catcher
584,1.0,6.02,20.0,0.0,Drew Millas,Catcher
588,10.0,4.43,80.0,0.0,Riley Adams,Catcher


Catcher/First Basemen Stats filled with zero if null

Team: 0
Name: 0
Pos: 0
Status: 0
ProjectedOpeningDayRole: 0
Age: 0
ServiceTime: 2
Is40Man: 0
Options: 181
Proj_PA: 82
Proj_IP: 599
Proj_PT: 82
first_name: 0
last_name: 0
player_id: 0
full_name: 0
position_code: 0
position_name: 0
position_abbr: 0
team_id: 0
team_name: 0
league_id: 0
league: 0
num_teams: 0
gamesPlayed: 0
runs: 0
doubles: 0
triples: 0
homeRuns: 0
baseOnBalls: 0
hits: 0
hitByPitch: 0
avg: 0
atBats: 0
obp: 0
slg: 0
ops: 0
stolenBases: 0
plateAppearances: 0
totalBases: 0
rbi: 0
sacBunts: 0
atBatsPerHomeRun: 0
strikeOuts: 0
intentionalWalks: 0
caughtStealing: 0
stolenBasePercentage: 0
caughtStealingPercentage: 0
groundIntoDoublePlay: 0
sacFlies: 0
babip: 0
groundOuts: 0
airOuts: 0
numberOfPitches: 0
leftOnBase: 0
groundOutsToAirouts: 0
catchersInterference: 0
first_name: 0
last_name: 0
gamesStarted: 0
assists: 0
putOuts: 0
errors: 0
chances: 0
fielding: 0
rangeFactorPerGame: 0
rangeFactorPer9Inn: 0
innings: 0
games: 0
dou

,Name,Team,Proj_PA,ServiceTime,Is40Man,Status
16,Aramis Garcia,ARI,NaN,3.060,N,NRI
17,Óscar Mercado,ARI,NaN,3.031,N,NRI
18,Jacob Amaya,ARI,NaN,0.118,N,NRI
31,Brian Serven,ATH,NaN,1.089,N,NRI
32,Michael Stefanic,ATH,NaN,1.092,N,NRI
...,...,...,...,...,...,...
556,Andrew Velazquez,TEX,NaN,3.033,N,NRI
557,Richie Martin,TEX,NaN,2.138,N,NRI
573,Carlos Mendoza,TOR,NaN,NaN,N,NRI
590,Tres Barrera,WSN,NaN,1.032,N,NRI


Offense Nulls Remaining:
Proj_PT    0
Proj_PA    0
Proj_IP    0
dtype: int64
Verfify Sums add to 1
Pitching
Team
ARI    1.0
ATH    1.0
ATL    1.0
BAL    1.0
BOS    1.0
Name: P_Player_Weight, dtype: float64
Offense
Team
ARI    1.0
ATH    1.0
ATL    1.0
BAL    1.0
BOS    1.0
Name: Player_Weight, dtype: float64
--- Offense Weight Check ---
✅ All Offense team weights sum to 1.0 (within buffer).
--- Pitching Weight Check ---
✅ All Pitching team weights sum to 1.0 (within buffer).


## Null Counts

In [748]:
print('Length of Merged Pitching: ', len(merged_pitching_clean))
for key, value in  merged_pitching_clean.isnull().sum().items():
    print(f"{key}: {value}")
print()
for key, value in  merged_offense_clean.isnull().sum().items():
    print(f"{key}: {value}")

Length of Merged Pitching:  815
Team: 0
Name: 0
Pos: 0
Status: 2
ProjectedOpeningDayRole: 0
Age: 0
ServiceTime: 0
Is40Man: 0
Options: 217
Proj_PA: 814
Proj_IP: 0
Proj_PT: 142
P_first_name: 0
P_last_name: 0
P_player_id: 0
P_full_name: 0
P_position_code: 0
P_position_name: 0
P_position_abbr: 0
P_team_id: 0
P_team_name: 0
P_league_id: 0
P_league: 0
P_num_teams: 0
P_gamesPlayed: 0
P_gamesStarted: 0
P_groundOuts: 0
P_airOuts: 0
P_runs: 0
P_doubles: 0
P_triples: 0
P_homeRuns: 0
P_strikeOuts: 0
P_baseOnBalls: 0
P_intentionalWalks: 0
P_hits: 0
P_hitByPitch: 0
P_avg: 0
P_atBats: 0
P_obp: 0
P_slg: 0
P_ops: 0
P_caughtStealing: 0
P_stolenBases: 0
P_stolenBasePercentage: 0
P_caughtStealingPercentage: 0
P_groundIntoDoublePlay: 0
P_numberOfPitches: 0
P_era: 0
P_inningsPitched: 0
P_wins: 0
P_losses: 0
P_saves: 0
P_saveOpportunities: 0
P_holds: 0
P_blownSaves: 0
P_earnedRuns: 0
P_whip: 0
P_battersFaced: 0
P_outs: 0
P_gamesPitched: 0
P_completeGames: 0
P_shutouts: 0
P_strikes: 0
P_strikePercentage: 0
P_

## Load Training Data
- Compare columsn of training data to the player career data

In [750]:
team_offense=pd.read_csv(os.path.join(data_dir,'TeamWins_clean.csv'))
team_fielding=pd.read_csv(os.path.join(data_dir,'teamFielding1995-2025.csv'))
league_info=pd.read_csv(os.path.join(data_dir,'leagueInfo.csv'))
league_info=league_info[['CODE','division_id','league_id']]
league_info['CODE']=league_info['CODE'].apply(lambda x: 'ATH' if x=='OAK' else x)
merged_offense_clean=pd.merge(merged_offense_clean.drop(columns=['league_id']),league_info, left_on='Team', right_on='CODE')
merged_pitching_clean=pd.merge(merged_pitching_clean.drop(columns=['P_league_id']),league_info, left_on='Team', right_on='CODE')

final_wins=pd.merge(team_offense, team_fielding.drop(columns=['LEAGUE', 'DIVISION', 'WINS','G']), on=['TEAM','Year'])
print('Columns in Training Data: ', final_wins.columns)
print()
print('Columns in 2026 Offensive Data: ', merged_offense_clean.columns)
print()
print('Columns in 2026 Pitching Data: ', merged_pitching_clean.columns)


Columns in Training Data:  Index(['TEAM', 'LEAGUE', 'DIVISION', 'WINS', 'Year', 'G', 'AB', 'R', 'H', '2B',
       '3B', 'HR', 'RBI', 'BB', 'SO', 'SB', 'CS', 'AVG', 'OBP', 'SLG', 'OPS',
       'W_pitch', 'L_pitch', 'ERA_pitch', 'G_pitch', 'GS_pitch', 'CG_pitch',
       'SHO_pitch', 'SV_pitch', 'SVO_pitch', 'IP_pitch', 'H_pitch', 'R_pitch',
       'ER_pitch', 'HR_pitch', 'HB_pitch', 'BB_pitch', 'SO_pitch',
       'WHIP_pitch', 'AVG_pitch', 'errors', 'fielding_pct', 'putOuts',
       'assists', 'chances', 'doublePlays', 'triplePlays',
       'rangeFactorPerGame', 'rangeFactorPer9Inn', 'innings', 'passedBall',
       'wildPitches', 'throwingErrors', 'caughtStealing', 'stolenBasesAllowed',
       'stolenBasePercentage', 'catchersInterference', 'pickoffs'],
      dtype='object')

Columns in 2026 Offensive Data:  Index(['Team', 'Name', 'Pos', 'Status', 'ProjectedOpeningDayRole', 'Age',
       'ServiceTime', 'Is40Man', 'Options', 'Proj_PA', 'Proj_IP', 'Proj_PT',
       'first_name', 'last_name

## Column Renaming — Align Inference Data to Training Schema
Rename columns in the 2026 offense and pitching dataframes to match the
column names used in the historical training data.

In [752]:
# Hitting inference → training
offense_to_training = {
    'Team':               'TEAM',
    'league_id':          'LEAGUE',
    'division_id':        'DIVISION',
    'gamesPlayed':        'G',
    'atBats':             'AB',
    'runs':               'R',
    'hits':               'H',
    'doubles':            '2B',
    'triples':            '3B',
    'homeRuns':           'HR',
    'rbi':                'RBI',
    'baseOnBalls':        'BB',
    'strikeOuts':         'SO',
    'stolenBases':        'SB',
    'caughtStealing':     'CS',
    'avg':                'AVG',
    'obp':                'OBP',
    'slg':                'SLG',
    'ops':                'OPS',
    'fielding':           'fielding_pct',          # inference 'fielding' → training 'fielding_pct'
    'errors':             'errors',
    'putOuts':            'putOuts',
    'assists':            'assists',
    'chances':            'chances',
    'doublePlays':        'doublePlays',
    'triplePlays':        'triplePlays',
    'rangeFactorPerGame': 'rangeFactorPerGame',
    'rangeFactorPer9Inn': 'rangeFactorPer9Inn',
    'innings':            'innings',
    'passedBall':         'passedBall',
    'wildPitches':        'wildPitches',
    'throwingErrors':     'throwingErrors',
    'catchersInterference': 'catchersInterference',
    'stolenBasePercentage': 'stolenBasePercentage',
    'pickoffs':           'pickoffs',
}

# Pitching inference → training
pitching_to_training = {
    'Team':                 'TEAM',
    'league_id':            'LEAGUE',
    'division_id':          'DIVISION',
    'P_wins':               'W_pitch',
    'P_losses':             'L_pitch',
    'P_era':                'ERA_pitch',
    'P_gamesPitched':       'G_pitch',
    'P_gamesStarted':       'GS_pitch',
    'P_completeGames':      'CG_pitch',
    'P_shutouts':           'SHO_pitch',
    'P_saves':              'SV_pitch',
    'P_saveOpportunities':  'SVO_pitch',
    'P_inningsPitched':     'IP_pitch',
    'P_hits':               'H_pitch',
    'P_runs':               'R_pitch',
    'P_earnedRuns':         'ER_pitch',
    'P_homeRuns':           'HR_pitch',
    'P_hitBatsmen':         'HB_pitch',
    'P_baseOnBalls':        'BB_pitch',
    'P_strikeOuts':         'SO_pitch',
    'P_stolenBases':        'stolenBasesAllowed',   # stolen bases allowed by pitcher
    'P_whip':               'WHIP_pitch',
    'P_avg':                'AVG_pitch',
}

final_2026_offense = merged_offense_clean.rename(columns=offense_to_training)
final_2026_pitching = merged_pitching_clean.rename(columns=pitching_to_training)

In [753]:
final_2026_offense['fielding_pct'] = final_2026_offense['fielding_pct'].replace(0.0, np.nan)

print("=== Zero Value Counts per Column ===")
zero_counts = (final_2026_offense == 0.0).sum()
zero_counts = zero_counts[zero_counts > 0].sort_values(ascending=False)
print(zero_counts)

print("\n=== NaN Value Counts per Column ===")
nan_counts = final_2026_offense.isna().sum()
nan_counts = nan_counts[nan_counts > 0].sort_values(ascending=False)
print(nan_counts)
print(final_2026_offense.dtypes.to_string())

=== Zero Value Counts per Column ===
Proj_IP                 599
triplePlays             580
pickoffs                546
passedBall              512
wildPitches             505
catcherERA              504
catchersInterference    483
throwingErrors          277
intentionalWalks        249
sacBunts                221
doublePlays             176
errors                  155
3B                      124
CS                      117
assists                 113
gamesStarted             89
Proj_PA                  82
Proj_PT                  82
Player_Weight            82
sacFlies                 80
SB                       67
putOuts                  65
hitByPitch               54
rangeFactorPerGame       53
chances                  53
HR                       33
groundIntoDoublePlay     30
2B                       14
RBI                       7
BB                        6
airOuts                   5
R                         3
ServiceTime               3
innings                   2
leftOnBase 

## Derive Missing Hitting Columns for Training Data
Calculate columns present in the inference data but missing from the historical
training data. Exact calculations are always applied (`totalBases`, `stolenBasePercentage`,
`caughtStealingPercentage`, `atBatsPerHomeRun`). Approximate calculations (`babip`,
`plateAppearances`) are opt-in via `use_approx=True` since they rely on missing
components like sacrifice flies and hit-by-pitch.

In [755]:
def derive_hitting_columns(df,use_approx=False):
    df = df.copy()


    # Total bases: singles + 2x doubles + 3x triples + 4x HRs
    df['totalBases'] = (df['H'] - df['2B'] - df['3B'] - df['HR']) + \
                       (2 * df['2B']) + (3 * df['3B']) + (4 * df['HR'])

    # Stolen base percentage
    df['stolenBasePercentage'] = (df['SB'] / (df['SB'] + df['CS'])).replace([np.inf, -np.inf], 0).fillna(0)

    # Caught stealing percentage
    df['caughtStealingPercentage'] = (df['CS'] / (df['SB'] + df['CS'])).replace([np.inf, -np.inf], 0).fillna(0)

    # At bats per home run
    df['atBatsPerHomeRun'] = (df['AB'] / df['HR']).replace([np.inf, -np.inf], 0).fillna(0)

    if use_approx:
        # BABIP — approximate without sacFlies
        df['babip'] = ((df['H'] - df['HR']) / (df['AB'] - df['SO'] - df['HR'])).replace([np.inf, -np.inf], 0).fillna(0)
    
        # Plate appearances — approximate without HBP/SF/SH
        df['plateAppearances'] = df['AB'] + df['BB']

    return df

final_wins = derive_hitting_columns(final_wins)

## Training Data Column statistics for total type columns (RBI, Hits, Walks, etc)

In [757]:
tally_cols = ['G', 'AB', 'R', 'H', '2B', '3B', 'HR', 'RBI', 'BB', 'SO', 'SB', 'CS', 'assists', 'putOuts', 'errors', 'chances', 'doublePlays', 'triplePlays', 'throwingErrors', 'passedBall', 'wildPitches', 'pickoffs', 'catchersInterference']
league_stats = final_wins[tally_cols].mean().to_dict()
league_stats

{'G': 161.40044742729307,
 'AB': 5510.230425055928,
 'R': 743.3937360178971,
 'H': 1424.2494407158836,
 '2B': 282.18791946308727,
 '3B': 28.568232662192393,
 'HR': 175.03803131991052,
 'RBI': 707.8881431767338,
 'BB': 526.1040268456376,
 'SO': 1177.8557046979865,
 'SB': 96.92281879194631,
 'CS': 37.446308724832214,
 'assists': 1584.3366890380314,
 'putOuts': 4315.6767337807605,
 'errors': 100.04250559284117,
 'chances': 6000.055928411633,
 'doublePlays': 144.53691275167785,
 'triplePlays': 0.12639821029082773,
 'throwingErrors': 41.58612975391499,
 'passedBall': 10.885906040268456,
 'wildPitches': 53.70022371364653,
 'pickoffs': 1.9015659955257271,
 'catchersInterference': 1.1431767337807606}

## Team-Level Hitting & Fielding Projections (`aggregateHitting`)
Aggregate player-level 2026 stats to team-level projections using projected
plate appearances as weights. Tally stats are normalized per game, weighted,
summed by team, and scaled to a full 162-game season. Rate stats are weighted
averages only. All tally columns are mean-shifted to match historical league
averages before output. Final output is filtered to only the columns present
in the training data.

In [759]:
class aggregateHitting:
    def __init__(self, df, historical_stats_dict):
        self.df = df.copy()
        self.historical_stats = historical_stats_dict
        self.training_columns = [
            'TEAM', 'LEAGUE', 'DIVISION', 'WINS', 'Year', 'G', 'AB', 'R', 'H', '2B',
            '3B', 'HR', 'RBI', 'BB', 'SO', 'SB', 'CS', 'AVG', 'OBP', 'SLG', 'OPS',
            'W_pitch', 'L_pitch', 'ERA_pitch', 'G_pitch', 'GS_pitch', 'CG_pitch',
            'SHO_pitch', 'SV_pitch', 'SVO_pitch', 'IP_pitch', 'H_pitch', 'R_pitch',
            'ER_pitch', 'HR_pitch', 'HB_pitch', 'BB_pitch', 'SO_pitch',
            'WHIP_pitch', 'AVG_pitch', 'errors', 'fielding_pct', 'putOuts',
            'assists', 'chances', 'doublePlays', 'triplePlays',
            'rangeFactorPerGame', 'rangeFactorPer9Inn', 'innings', 'passedBall',
            'wildPitches', 'throwingErrors', 'caughtStealing', 'stolenBasesAllowed',
            'stolenBasePercentage', 'catchersInterference', 'pickoffs',
            'totalBases', 'caughtStealingPercentage', 'atBatsPerHomeRun'
        ]

        self.hitting_tallies = [
            'G', 'AB', 'R', 'H', '2B', '3B', 'HR', 'RBI', 'BB', 'SO', 'SB', 'CS',
            'hitByPitch', 'intentionalWalks', 'plateAppearances', 'totalBases',
            'sacBunts', 'sacFlies', 'groundIntoDoublePlay', 'groundOuts', 'airOuts',
            'numberOfPitches', 'leftOnBase',
        ]

        self.fielding_tallies = [
            'gamesStarted', 'assists', 'putOuts', 'errors', 'chances',
            'doublePlays', 'triplePlays', 'throwingErrors', 'passedBall',
            'wildPitches', 'pickoffs', 'catchersInterference',
        ]

        self.rate_columns = [
            'AVG', 'OBP', 'SLG', 'OPS', 'babip',
            'stolenBasePercentage', 'caughtStealingPercentage',
            'atBatsPerHomeRun', 'groundOutsToAirouts',
            'fielding_pct', 'rangeFactorPerGame', 'rangeFactorPer9Inn',
            'catcherERA',
        ]

        self.all_tallies = self.hitting_tallies + self.fielding_tallies

        # Tally columns and Player_Weight/G: coerce and fillna(0) — zero is meaningful
        cols_to_fix = self.all_tallies + ['Player_Weight', 'G']
        for col in cols_to_fix:
            if col in self.df.columns:
                self.df[col] = pd.to_numeric(self.df[col], errors='coerce').fillna(0)

        # Rate columns: coerce but preserve NaN — zero is NOT meaningful here
        for col in self.rate_columns:
            if col in self.df.columns:
                self.df[col] = pd.to_numeric(self.df[col], errors='coerce')

        self.add_per_game_columns()

    def add_per_game_columns(self):
        for col in self.all_tallies:
            if col in self.df.columns:
                self.df[f"{col}_per_game"] = (
                    self.df[col] / self.df['G']
                ).replace([np.inf, -np.inf], 0).fillna(0)

    def calculate_team_projections(self, games_per_season=162, slots=10):
        # 1. Weight the tally rates
        for col in self.all_tallies:
            if f"{col}_per_game" in self.df.columns:
                self.df[f'w_tally_{col}'] = self.df[f"{col}_per_game"] * self.df['Player_Weight']

        # 2. Weight rate columns with per-column weight renormalization
        # Weights are renormalized over only players with valid (non-NaN) data
        # so the weighted average is always properly calibrated to sum to 1.0
        for col in self.rate_columns:
            if col in self.df.columns:
                valid_mask = self.df[col].notna()
                valid_weight_sum = self.df.groupby('TEAM')['Player_Weight'].transform(
                    lambda x: x[valid_mask.loc[x.index]].sum()
                )
                renorm_weight = self.df['Player_Weight'] / valid_weight_sum
                # NaN × renorm_weight = NaN, naturally excluded from groupby sum
                self.df[f'w_rate_{col}'] = self.df[col] * renorm_weight

        # 3. Aggregation
        tally_cols = [c for c in self.df.columns if c.startswith('w_tally_')]
        rate_cols  = [c for c in self.df.columns if c.startswith('w_rate_')]

        agg_map = {col: 'sum' for col in (tally_cols + rate_cols)}
        for cat in ['LEAGUE', 'DIVISION']:
            if cat in self.df.columns:
                agg_map[cat] = 'first'

        team_df = self.df.groupby('TEAM').agg(agg_map)

        # 4. Scale tallies to full season
        final_tallies = team_df[tally_cols] * games_per_season * slots
        final_tallies.columns = [c.replace('w_tally_', '') for c in final_tallies.columns]

        # Mean shift to match historical averages
        for col, avg in self.historical_stats.items():
            if col in final_tallies.columns:
                avg_diff = final_tallies[col].mean() - avg
                final_tallies[col] = final_tallies[col] - avg_diff

        # Rates: weighted sum is already the calibrated average
        final_rates = team_df[rate_cols]
        final_rates.columns = [c.replace('w_rate_', '') for c in final_rates.columns]

        result = pd.concat([final_tallies, final_rates], axis=1)

        for cat in ['LEAGUE', 'DIVISION']:
            if cat in team_df.columns:
                result[cat] = team_df[cat]

        return result.reset_index()[
            ['TEAM'] + [col for col in result.columns
                        if col in self.training_columns and col != 'TEAM']
        ]
processor = aggregateHitting(final_2026_offense,league_stats)
team_2026_data = processor.calculate_team_projections()
team_2026_data
#UPDATED

,TEAM,G,AB,R,H,2B,3B,HR,RBI,BB,...,SLG,OPS,stolenBasePercentage,caughtStealingPercentage,atBatsPerHomeRun,fielding_pct,rangeFactorPerGame,rangeFactorPer9Inn,LEAGUE,DIVISION
0,ARI,161.400447,5358.584710,742.507483,1381.504121,285.986766,37.826227,149.085404,683.664489,541.003815,...,0.408129,0.727250,0.735633,0.264443,33.437382,0.985304,3.994995,5.020544,NL,W
1,ATH,161.400447,5486.866497,719.573425,1445.170120,292.269934,23.451898,204.606404,705.338361,476.929123,...,0.443505,0.763730,0.748531,0.251469,30.784301,0.989430,3.947981,5.081619,AL,W
2,ATL,161.400447,5654.506822,826.738549,1500.045361,300.302677,27.157073,221.567122,800.883013,548.980464,...,0.449963,0.778063,0.756605,0.243395,26.178328,0.986971,4.086966,4.755764,NL,E
3,BAL,161.400447,5719.285271,779.035391,1409.776196,286.234251,28.948109,215.289091,761.222239,608.419793,...,0.426929,0.746917,0.701788,0.298212,25.411152,0.988706,4.516616,5.991224,AL,E
4,BOS,161.400447,5494.415728,773.347540,1450.519271,338.017873,31.216097,157.598288,689.254169,496.388850,...,0.429770,0.755779,0.685314,0.314686,33.755360,0.983474,3.733529,4.670216,AL,E
5,CHC,161.400447,5504.884950,786.554373,1439.218554,289.564318,34.676890,182.908802,768.806899,590.673556,...,0.435064,0.765893,0.785977,0.214023,29.791936,0.988848,2.823678,3.214871,NL,C
6,CHW,161.400447,5375.886472,673.985474,1357.950798,249.543092,16.672004,146.520644,631.649622,491.725301,...,0.393167,0.705827,0.697403,0.302728,43.449432,0.984999,4.269690,5.155913,AL,C
7,CIN,161.400447,5610.810904,781.834176,1427.022832,260.954785,32.849272,189.234184,702.277846,524.647334,...,0.422852,0.739644,0.745312,0.254782,29.678574,0.973544,3.176676,4.393541,NL,C
8,CLE,161.400447,5196.846898,662.824026,1274.139299,283.140073,26.020250,148.152591,637.294505,535.814955,...,0.400088,0.710739,0.791511,0.208489,36.583822,0.985186,4.219630,5.256058,AL,C
9,COL,161.400447,5359.668065,660.001921,1339.507339,255.439380,44.189146,132.215228,597.593871,393.846996,...,0.395207,0.697594,0.709973,0.290042,48.414407,0.983871,3.528604,4.693898,NL,W


## Derive Missing Pitching Rate Columns for Training Data
Calculate per-9-inning rates and ratio stats present in the inference data
but missing from the historical training data. Derived from existing `_pitch`
tally columns — strikeout/walk ratio, win percentage, and per-9 rates for
strikeouts, walks, hits, home runs, and runs scored.

In [761]:
def derive_pitching_columns(df):
    df = df.copy()
    
    # Ratio stats
    df['P_strikeoutWalkRatio'] = df['SO_pitch'] / df['BB_pitch'].replace(0, np.nan)
    
    # Win percentage
    df['P_winPercentage'] = df['W_pitch'] / (df['W_pitch'] + df['L_pitch']).replace(0, np.nan)
    
    # Save percentage
    df['SV_pct'] = df['SV_pitch'] / df['SVO_pitch'].replace(0, np.nan)
    
    # Per 9 inning rates
    ip = df['IP_pitch'].replace(0, np.nan)
    df['P_strikeoutsPer9Inn'] = (df['SO_pitch'] / ip) * 9
    df['P_walksPer9Inn']      = (df['BB_pitch'] / ip) * 9
    df['P_hitsPer9Inn']       = (df['H_pitch']  / ip) * 9
    df['P_homeRunsPer9']      = (df['HR_pitch'] / ip) * 9
    df['P_runsScoredPer9']    = (df['R_pitch']  / ip) * 9
    
    # Fill any infs from division
    rate_cols = ['P_strikeoutWalkRatio', 'P_winPercentage', 'SV_pct',
                 'P_strikeoutsPer9Inn', 'P_walksPer9Inn', 'P_hitsPer9Inn', 
                 'P_homeRunsPer9', 'P_runsScoredPer9']
    df[rate_cols] = df[rate_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    
    return df

final_wins = derive_pitching_columns(final_wins)
final_2026_pitching['SV_pct'] = final_2026_pitching['SV_pitch'] / final_2026_pitching['SVO_pitch'].replace(0, np.nan)

### Pitching Data Audit

In [763]:
print("=== Zero Value Counts per Column ===")
zero_counts = (final_2026_pitching == 0.0).sum()
zero_counts = zero_counts[zero_counts > 0].sort_values(ascending=False)
print(zero_counts)

print("\n=== NaN Value Counts per Column ===")
nan_counts = final_2026_pitching.isna().sum()
nan_counts = nan_counts[nan_counts > 0].sort_values(ascending=False)
print(nan_counts)

print("\n=== dtypes ===")
print(final_2026_pitching.dtypes.to_string())

=== Zero Value Counts per Column ===
SHO_pitch                   735
CG_pitch                    706
P_catchersInterference      578
P_balks                     458
SV_pitch                    435
P_pickoffs                  417
P_blownSaves                390
SVO_pitch                   323
P_holds                     275
P_intentionalWalks          272
GS_pitch                    262
P_inheritedRunnersScored    249
P_sacBunts                  240
P_triples                   206
P_caughtStealing            188
P_inheritedRunners          168
P_gamesFinished             144
P_Player_Weight             142
Proj_IP                     142
P_sacFlies                  119
P_wildPitches               112
SV_pct                      112
W_pitch                     102
L_pitch                      93
HB_pitch                     92
P_hitByPitch                 92
stolenBasesAllowed           68
P_groundIntoDoublePlay       57
HR_pitch                     44
P_homeRunsPer9               29
P_d

### Save Percentage Vetting — Zero `SV_pct` Closers

Before running the pitching aggregation, we verify that any closer with `SV_pct = 0.0` is not a data artifact. A zero save percentage for a closer means either:

- They have real save opportunities on record and converted none of them — a legitimate (if rare) career line
- They have no save data at all and defaulted to zero — a data quality issue that would corrupt the team's projected save percentage

Filtering to `ProjectedOpeningDayRole == 'Closer'` isolates the only cases that matter, since non-closers with zero save percentage are expected and handled by the role-based weighting in `calculate_closer_rates`.

The investigation found a single closer with `SV_pct = 0.0` — Justin Sterner (ATH) with 6 save opportunities and 0 conversions. This is real data, not a placeholder. No action required.

In [765]:
zero_sv_pct=final_2026_pitching[final_2026_pitching['SV_pct'] == 0.0][['Name','IP_pitch', 'TEAM', 'Pos', 'ProjectedOpeningDayRole', 'SV_pct', 'SV_pitch', 'SVO_pitch', 'P_Player_Weight']].sort_values('P_Player_Weight', ascending=False)
display(zero_sv_pct[zero_sv_pct['ProjectedOpeningDayRole']=='Closer'])

,Name,IP_pitch,TEAM,Pos,ProjectedOpeningDayRole,SV_pct,SV_pitch,SVO_pitch,P_Player_Weight
39,Justin Sterner,69.0,ATH,RP,Closer,0.0,0,6.0,0.044704


### ERA Vetting — Zero `ERA_pitch` Pitchers

A career ERA of `0.0` is not a realistic statistic for any pitcher with meaningful MLB experience — it indicates a tiny sample size where the pitcher happened to allow no earned runs. Left uncorrected, these pitchers would drag down their team's weighted ERA projection.

We identify all pitchers with `ERA_pitch = 0.0` and sort by `P_Player_Weight` to find cases where the zero would actually influence the team aggregate. The concerning cases are pitchers with both a non-trivial weight and zero ERA — all of which trace back to extremely small career samples (1–4 IP).

**Resolution:** The 30 IP threshold applied in `aggregatePitching.__init__` nulls out rate columns for all pitchers below the minimum innings threshold. This means zero-ERA pitchers like Barco (3 IP), Raquet (2 IP), and Espada (4 IP) have their `ERA_pitch` set to `NaN` before aggregation, and their weight is redistributed to pitchers with meaningful career samples via the per-column renormalization in `calculate_team_projections`. No manual intervention required.

In [767]:
zero_era = final_2026_pitching[final_2026_pitching['ERA_pitch'] == 0.0][['Name', 'TEAM', 'ProjectedOpeningDayRole', 'ERA_pitch', 'IP_pitch', 'P_Player_Weight']]
print(f"Players with ERA = 0.0: {len(zero_era)}")
display(zero_era.sort_values('P_Player_Weight', ascending=False))

Players with ERA = 0.0: 8


,Name,TEAM,ProjectedOpeningDayRole,ERA_pitch,IP_pitch,P_Player_Weight
583,Hunter Barco,PIT,Rotation/Bullpen Candidate,0.0,3.0,0.056942
699,Nick Raquet,STL,Optioned,0.0,2.0,0.021362
110,Jose Espada,BAL,Bullpen Candidate,0.0,4.0,0.010592
540,Dom Hamel,NYY,Bullpen Candidate,0.0,1.0,0.004911
90,Austin Pope,ATL,Reassigned,0.0,2.0,0.000000
213,Hagen Danner,CIN,Bullpen Candidate,0.0,0.1,0.000000
319,Christian Roa,HOU,Bullpen Candidate,0.0,3.0,0.000000
541,Adam Kloffenstein,NYY,Reassigned,0.0,1.0,0.000000


### Ensure Adequate Closer Data

In [769]:
save_roles = final_2026_pitching[
    final_2026_pitching['ProjectedOpeningDayRole'].isin(['Closer', 'Setup Man'])
]

team_save_check = save_roles.groupby('TEAM').agg(
    n_closers   = ('ProjectedOpeningDayRole', lambda x: (x == 'Closer').sum()),
    n_setup_men = ('ProjectedOpeningDayRole', lambda x: (x == 'Setup Man').sum()),
    avg_SV_pct  = ('SV_pct', 'mean')
).reset_index()

team_save_check['has_closer_or_setup'] = (team_save_check['n_closers'] + team_save_check['n_setup_men']) > 0

print(f"Teams with no Closer or Setup Man: {(~team_save_check['has_closer_or_setup']).sum()}")
print(f"Teams with only Setup Men:         {((team_save_check['n_closers'] == 0) & (team_save_check['n_setup_men'] > 0)).sum()}")
print(f"Teams with only Closers:           {((team_save_check['n_closers'] > 0) & (team_save_check['n_setup_men'] == 0)).sum()}")
print(f"Teams with both:                   {((team_save_check['n_closers'] > 0) & (team_save_check['n_setup_men'] > 0)).sum()}")
print()
display(team_save_check)

Teams with no Closer or Setup Man: 0
Teams with only Setup Men:         0
Teams with only Closers:           3
Teams with both:                   27



,TEAM,n_closers,n_setup_men,avg_SV_pct,has_closer_or_setup
0,ARI,2,2,0.528749,True
1,ATH,3,0,0.463768,True
2,ATL,1,2,0.610474,True
3,BAL,1,2,0.620899,True
4,BOS,1,2,0.598921,True
5,CHC,1,2,0.582011,True
6,CHW,1,2,0.674603,True
7,CIN,1,2,0.391667,True
8,CLE,1,2,0.481717,True
9,COL,1,2,0.444444,True


### IP Threshold Investigation — Identifying Unreliable Rate Stats

Career rate statistics like ERA are only meaningful when derived from a sufficient sample of innings. A pitcher with 2 career MLB innings who allowed zero earned runs carries a 0.00 ERA, but this tells us nothing about their true ability — it is noise masquerading as signal. In a weighted team projection, even a low-weight pitcher with an extreme ERA can meaningfully distort the team average.

**Cell 1: Pitchers below threshold with non-zero weight**

We first identify all pitchers with fewer than 10 career IP who still carry a non-zero projected playing time weight. These are the cases where unreliable rate stats would actually influence the team projection. The output confirmed several pitchers with extreme ERAs (0.00, 27.00, 19.50) driven entirely by 1–6 innings of career data.

This motivated raising the threshold to **30 IP** — the point at which career ERA begins to stabilize as a meaningful signal — and nulling out all rate columns below that threshold rather than dropping the pitchers entirely. Their tally contributions (strikeouts, walks, hits) are still counted; only their rate stats are excluded from the weighted average.

**Cell 2: Closer and Setup Man coverage after applying the 30 IP threshold**

Applying the threshold to the full roster raised a secondary concern: if a team's designated closer has fewer than 30 career innings, the save percentage projection loses its primary input. We verify that every team retains at least one closer or setup man above the threshold, and check whether the 70/30 closer/setup man weighting in `calculate_closer_rates` still has valid inputs for all 30 teams.

In [771]:
ip_threshold = 10
print("Pitchers below IP threshold with non-zero weight:")
display(final_2026_pitching[
    (final_2026_pitching['IP_pitch'] < ip_threshold) & 
    (final_2026_pitching['P_Player_Weight'] > 0)
][['Name', 'TEAM', 'ProjectedOpeningDayRole', 'ERA_pitch', 'IP_pitch', 'P_Player_Weight']].sort_values('P_Player_Weight', ascending=False))

Pitchers below IP threshold with non-zero weight:


,Name,TEAM,ProjectedOpeningDayRole,ERA_pitch,IP_pitch,P_Player_Weight
783,Foster Griffin,WSN,Starting Rotation,6.75,8.0,0.070758
583,Hunter Barco,PIT,Rotation/Bullpen Candidate,0.0,3.0,0.056942
604,Bradgley Rodriguez,SDP,Middle Reliever,1.17,7.2,0.031688
400,Kyle Hurt,LAD,Rotation/Bullpen Candidate,1.04,8.2,0.030506
453,Craig Yoho,MIL,Bullpen Candidate,7.27,8.2,0.027097
215,Chase Petty,CIN,Optioned,19.50,6.0,0.025594
584,Cam Sanders,PIT,Bullpen Candidate,8.10,6.2,0.025592
72,Hayden Harris,ATL,Middle Reliever,3.38,2.2,0.025240
237,Doug Nikhazy,CLE,Optioned,13.50,4.0,0.021711
699,Nick Raquet,STL,Optioned,0.0,2.0,0.021362


#### Ensure adequate closer data after discounting players with less than 30 Innings pitched

In [773]:
save_roles = final_2026_pitching[
    final_2026_pitching['ProjectedOpeningDayRole'].isin(['Closer', 'Setup Man']) &
    (final_2026_pitching['IP_pitch'] >= 30)
]

team_save_check = save_roles.groupby('TEAM').agg(
    n_closers   = ('ProjectedOpeningDayRole', lambda x: (x == 'Closer').sum()),
    n_setup_men = ('ProjectedOpeningDayRole', lambda x: (x == 'Setup Man').sum()),
    avg_SV_pct  = ('SV_pct', 'mean')
).reset_index()

# Teams with no closers or setup men at all after threshold
all_teams = final_2026_pitching['TEAM'].unique()
team_save_check = pd.DataFrame({'TEAM': all_teams}).merge(team_save_check, on='TEAM', how='left').fillna(0)
team_save_check['n_closers']   = team_save_check['n_closers'].astype(int)
team_save_check['n_setup_men'] = team_save_check['n_setup_men'].astype(int)
team_save_check['has_closer_or_setup'] = (team_save_check['n_closers'] + team_save_check['n_setup_men']) > 0

print(f"Teams with no Closer or Setup Man: {(~team_save_check['has_closer_or_setup']).sum()}")
print(f"Teams with only Setup Men:         {((team_save_check['n_closers'] == 0) & (team_save_check['n_setup_men'] > 0)).sum()}")
print(f"Teams with only Closers:           {((team_save_check['n_closers'] > 0) & (team_save_check['n_setup_men'] == 0)).sum()}")
print(f"Teams with both:                   {((team_save_check['n_closers'] > 0) & (team_save_check['n_setup_men'] > 0)).sum()}")
print()
display(team_save_check.sort_values('n_closers', ascending=True))

Teams with no Closer or Setup Man: 0
Teams with only Setup Men:         0
Teams with only Closers:           4
Teams with both:                   26



,TEAM,n_closers,n_setup_men,avg_SV_pct,has_closer_or_setup
14,LAD,1,2,0.707172,True
27,TEX,1,2,0.453571,True
24,SFG,1,2,0.354545,True
23,SEA,1,2,0.666309,True
22,SDP,1,2,0.577246,True
21,PIT,1,2,0.504395,True
20,PHI,1,2,0.696841,True
19,NYY,1,2,0.650530,True
18,NYM,1,2,0.727143,True
17,MIN,1,2,0.755949,True


## Stats From pitching tally-like columns (Hits allowed, Earned Runs, etc)

In [775]:
starter_tallies = [
                            'GS_pitch',
                            'CG_pitch',
                            'SHO_pitch',
                            'IP_pitch',
                        ]

closer_tallies = [
            'SV_pitch',
            'SVO_pitch',
        ]
        
general_tallies = [
            'W_pitch',
            'L_pitch',
            'G_pitch',
            'H_pitch',
            'R_pitch',
            'ER_pitch',
            'HR_pitch',
            'HB_pitch',
            'BB_pitch',
            'SO_pitch',
            'P_strikeoutWalkRatio',
            'P_winPercentage',
            'P_strikeoutsPer9Inn',
            'P_walksPer9Inn',
            'P_hitsPer9Inn',
            'P_homeRunsPer9',
            'P_runsScoredPer9',
        ]
pitching_tallies = starter_tallies + closer_tallies + general_tallies

league_stats_pitch=final_wins[pitching_tallies].mean().to_dict()
league_stats_pitch

{'GS_pitch': 161.40044742729307,
 'CG_pitch': 4.837807606263982,
 'SHO_pitch': 9.256152125279643,
 'IP_pitch': 1438.3350111856826,
 'SV_pitch': 40.62751677852349,
 'SVO_pitch': 58.710290827740494,
 'W_pitch': 80.68791946308725,
 'L_pitch': 80.68791946308725,
 'G_pitch': 161.40044742729307,
 'H_pitch': 1424.2494407158836,
 'R_pitch': 743.3937360178971,
 'ER_pitch': 682.9619686800895,
 'HR_pitch': 175.03803131991052,
 'HB_pitch': 57.83221476510067,
 'BB_pitch': 526.1040268456376,
 'SO_pitch': 1177.8557046979865,
 'P_strikeoutWalkRatio': 2.2860244107256293,
 'P_winPercentage': 0.4999858647875603,
 'P_strikeoutsPer9Inn': 7.366116720654002,
 'P_walksPer9Inn': 3.2941064741807753,
 'P_hitsPer9Inn': 8.914879482010408,
 'P_homeRunsPer9': 1.0957146768999777,
 'P_runsScoredPer9': 4.655152472401818}

## Team-Level Pitching Projections (`aggregatePitching`)
Aggregate player-level 2026 pitching stats to team-level projections using
projected innings pitched as weights. Tally stats are normalized per inning,
weighted, summed by team, and scaled to a full 1458-inning season (9 × 162).
Rate stats (ERA, WHIP, AVG, per-9 rates) are weighted averages only. Save
columns (`SV`, `SVO`) are handled separately — 70% weight to the designated
closer and 30% split among setup men — then merged back into the final result.
All tally columns are mean-shifted to match historical league averages.

In [777]:
import warnings

IP_THRESHOLD = 30

class aggregatePitching:
    def __init__(self, df, historical_stats_dict):
        self.df = df.copy()
        self.historical_stats = historical_stats_dict
        self.training_columns = ['TEAM', 'LEAGUE', 'DIVISION', 'WINS', 'Year', 'G', 'AB', 'R', 'H', '2B',
           '3B', 'HR', 'RBI', 'BB', 'SO', 'SB', 'CS', 'AVG', 'OBP', 'SLG', 'OPS',
           'W_pitch', 'L_pitch', 'ERA_pitch', 'G_pitch', 'GS_pitch', 'CG_pitch',
           'SHO_pitch', 'SV_pitch', 'SVO_pitch', 'IP_pitch', 'H_pitch', 'R_pitch',
           'ER_pitch', 'HR_pitch', 'HB_pitch', 'BB_pitch', 'SO_pitch',
           'WHIP_pitch', 'AVG_pitch', 'errors', 'fielding_pct', 'putOuts',
           'assists', 'chances', 'doublePlays', 'triplePlays',
           'rangeFactorPerGame', 'rangeFactorPer9Inn', 'innings', 'passedBall',
           'wildPitches', 'throwingErrors', 'caughtStealing', 'stolenBasesAllowed',
           'stolenBasePercentage', 'catchersInterference', 'pickoffs',
           'P_strikeoutWalkRatio', 'P_winPercentage', 'P_strikeoutsPer9Inn',
           'P_walksPer9Inn', 'P_hitsPer9Inn', 'P_homeRunsPer9',
           'P_runsScoredPer9', 'SV_pct']

        self.starter_tallies  = ['GS_pitch', 'CG_pitch', 'SHO_pitch', 'IP_pitch']
        self.closer_tallies   = ['SV_pitch', 'SVO_pitch']
        self.closer_rates     = ['SV_pct']
        self.general_tallies  = ['W_pitch', 'L_pitch', 'G_pitch', 'H_pitch', 'R_pitch',
                                  'ER_pitch', 'HR_pitch', 'HB_pitch', 'BB_pitch', 'SO_pitch']
        self.rate_columns     = ['ERA_pitch', 'WHIP_pitch', 'AVG_pitch', 'P_strikeoutWalkRatio',
                                  'P_winPercentage', 'P_strikeoutsPer9Inn', 'P_walksPer9Inn',
                                  'P_hitsPer9Inn', 'P_homeRunsPer9', 'P_runsScoredPer9']

        self.pitching_tallies = self.starter_tallies + self.closer_tallies + self.general_tallies
        self.all_tallies      = self.pitching_tallies

        # Tally columns: coerce and fillna(0) — zero is meaningful
        cols_to_fix = self.all_tallies + ['P_Player_Weight', 'P_gamesPlayed']
        for col in cols_to_fix:
            if col in self.df.columns:
                self.df[col] = pd.to_numeric(self.df[col], errors='coerce').fillna(0)

        # Rate columns: coerce but preserve NaN — zero is NOT meaningful
        # Also null out rates for pitchers below IP threshold — tiny samples corrupt team averages
        for col in self.rate_columns:
            if col in self.df.columns:
                self.df[col] = pd.to_numeric(self.df[col], errors='coerce')
                self.df.loc[self.df['IP_pitch'] < IP_THRESHOLD, col] = np.nan

        self.add_per_inning_columns()

    def add_per_inning_columns(self):
        for col in self.all_tallies:
            if col in self.df.columns:
                self.df[f"{col}_per_inning"] = (
                    self.df[col] / self.df['IP_pitch']
                ).replace([np.inf, -np.inf], 0).fillna(0)

    def calculate_team_projections(self, innings_per_season=1458):
        # 1. Weight the tally rates
        for col in self.all_tallies:
            if f"{col}_per_inning" in self.df.columns:
                self.df[f'w_tally_{col}'] = self.df[f"{col}_per_inning"] * self.df['P_Player_Weight']

        # 2. Weight rate columns with per-column weight renormalization
        # Weights renormalized over only players with valid (non-NaN) data
        for col in self.rate_columns:
            if col in self.df.columns:
                valid_mask = self.df[col].notna()
                valid_weight_sum = self.df.groupby('TEAM')['P_Player_Weight'].transform(
                    lambda x: x[valid_mask.loc[x.index]].sum()
                )
                renorm_weight = self.df['P_Player_Weight'] / valid_weight_sum
                self.df[f'w_rate_{col}'] = self.df[col] * renorm_weight

        # 3. Aggregate by team
        tally_cols = [c for c in self.df.columns if c.startswith('w_tally_')]
        rate_cols  = [c for c in self.df.columns if c.startswith('w_rate_')]

        agg_map = {col: 'sum' for col in (tally_cols + rate_cols)}
        for cat in ['LEAGUE', 'DIVISION']:
            if cat in self.df.columns:
                agg_map[cat] = 'first'

        team_df = self.df.groupby('TEAM').agg(agg_map)

        # 4. Scale tallies to full season
        final_tallies = team_df[tally_cols] * innings_per_season
        final_tallies.columns = [c.replace('w_tally_', '') for c in final_tallies.columns]

        # 5. Mean shift to match historical
        for col, avg in self.historical_stats.items():
            if col in final_tallies.columns:
                avg_diff = final_tallies[col].mean() - avg
                final_tallies[col] = final_tallies[col] - avg_diff

        # 6. Rates — weighted sum is already calibrated
        final_rates = team_df[rate_cols]
        final_rates.columns = [c.replace('w_rate_', '') for c in final_rates.columns]

        result = pd.concat([final_tallies, final_rates], axis=1)
        for cat in ['LEAGUE', 'DIVISION']:
            if cat in team_df.columns:
                result[cat] = team_df[cat]

        return result.reset_index()[['TEAM'] + [col for col in result.columns
                                         if col in self.training_columns and col != 'TEAM']]

    def calculate_closer_tallies(self, innings_per_season=1458):
        closer_roles = ['Closer', 'Setup Man']
        bullpen_df = self.df[
            self.df['ProjectedOpeningDayRole'].isin(closer_roles) &
            (self.df['IP_pitch'] >= IP_THRESHOLD)
        ].copy()

        def assign_save_weights(group):
            closers   = group['ProjectedOpeningDayRole'] == 'Closer'
            setup_men = group['ProjectedOpeningDayRole'] == 'Setup Man'
            n_closers, n_setup_men = closers.sum(), setup_men.sum()
            group = group.copy()
            group['save_weight'] = 0.0
            if n_closers > 0 and n_setup_men > 0:
                group.loc[closers,   'save_weight'] = 0.7 / n_closers
                group.loc[setup_men, 'save_weight'] = 0.3 / n_setup_men
            elif n_closers > 0:
                group.loc[closers,   'save_weight'] = 1.0 / n_closers
            elif n_setup_men > 0:
                group.loc[setup_men, 'save_weight'] = 1.0 / n_setup_men
            return group

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", DeprecationWarning)
            bullpen_df = bullpen_df.groupby('TEAM', group_keys=False).apply(assign_save_weights)

        for col in self.closer_tallies:
            if f"{col}_per_inning" in bullpen_df.columns:
                bullpen_df[f'w_tally_{col}'] = bullpen_df[f"{col}_per_inning"] * bullpen_df['save_weight']

        closer_tally_cols = [f'w_tally_{col}' for col in self.closer_tallies if f'w_tally_{col}' in bullpen_df.columns]
        closer_team = bullpen_df.groupby('TEAM')[closer_tally_cols].sum()

        final = closer_team * innings_per_season
        final.columns = [c.replace('w_tally_', '') for c in final.columns]

        for col, avg in self.historical_stats.items():
            if col in final.columns:
                avg_diff = final[col].mean() - avg
                final[col] = final[col] - avg_diff

        return final

    def calculate_closer_rates(self):
        closer_roles = ['Closer', 'Setup Man']
        bullpen_df = self.df[
            self.df['ProjectedOpeningDayRole'].isin(closer_roles) &
            (self.df['IP_pitch'] >= IP_THRESHOLD)
        ].copy()

        def assign_save_weights(group):
            closers   = group['ProjectedOpeningDayRole'] == 'Closer'
            setup_men = group['ProjectedOpeningDayRole'] == 'Setup Man'
            n_closers, n_setup_men = closers.sum(), setup_men.sum()
            group = group.copy()
            group['save_weight'] = 0.0
            if n_closers > 0 and n_setup_men > 0:
                group.loc[closers,   'save_weight'] = 0.7 / n_closers
                group.loc[setup_men, 'save_weight'] = 0.3 / n_setup_men
            elif n_closers > 0:
                group.loc[closers,   'save_weight'] = 1.0 / n_closers
            elif n_setup_men > 0:
                group.loc[setup_men, 'save_weight'] = 1.0 / n_setup_men
            return group

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", DeprecationWarning)
            bullpen_df = bullpen_df.groupby('TEAM', group_keys=False).apply(assign_save_weights)

        # Renormalize save_weights within each team to account for dropped pitchers
        bullpen_df['save_weight'] = bullpen_df.groupby('TEAM')['save_weight'].transform(
            lambda x: x / x.sum() if x.sum() > 0 else x
        )

        bullpen_df['w_SV_pct'] = bullpen_df['SV_pct'] * bullpen_df['save_weight']
        closer_rates = bullpen_df.groupby('TEAM')['w_SV_pct'].sum().rename('SV_pct')

        return closer_rates

    def get_team_projections(self):
        result        = self.calculate_team_projections()
        closer_result = self.calculate_closer_tallies()
        closer_rates  = self.calculate_closer_rates()

        result = result.set_index('TEAM')
        result.update(closer_result)
        result['SV_pct'] = closer_rates
        result = result.reset_index()

        return result[['TEAM'] + [col for col in self.training_columns
                                   if col in result.columns and col != 'TEAM']]
processor = aggregatePitching(final_2026_pitching, league_stats_pitch)
team_2026_pitching = processor.get_team_projections()
team_2026_pitching

,TEAM,LEAGUE,DIVISION,W_pitch,L_pitch,ERA_pitch,G_pitch,GS_pitch,CG_pitch,SHO_pitch,...,WHIP_pitch,AVG_pitch,P_strikeoutWalkRatio,P_winPercentage,P_strikeoutsPer9Inn,P_walksPer9Inn,P_hitsPer9Inn,P_homeRunsPer9,P_runsScoredPer9,SV_pct
0,ARI,NL,W,83.339713,81.351188,3.987940,174.291455,152.465134,4.016893,8.974072,...,1.258400,0.245175,3.052404,0.537793,8.648777,2.929540,8.391923,1.112864,4.322047,0.541853
1,ATH,AL,W,87.305884,79.801157,3.965551,114.702149,165.205912,4.401225,9.181792,...,1.256506,0.235565,2.780460,0.519208,8.863698,3.291342,8.024026,1.280204,4.308950,0.463768
2,ATL,NL,E,85.138147,73.638036,3.828411,215.274539,140.257448,5.694245,9.189761,...,1.230213,0.233770,3.296652,0.541569,9.474349,3.150920,7.913798,1.052009,4.107966,0.752892
3,BAL,AL,E,77.797533,83.776288,4.101673,60.682982,183.284250,5.136824,9.470305,...,1.269262,0.245365,3.019458,0.483813,8.747100,2.990752,8.428449,1.189018,4.384539,0.716904
4,BOS,AL,E,81.351166,86.199920,3.739814,100.406711,171.188235,6.165283,10.407734,...,1.254033,0.233946,3.024406,0.502718,9.357155,3.333446,7.948840,0.966830,4.148805,0.753567
5,CHC,NL,C,84.282613,72.452409,3.947761,157.441953,162.867464,4.457477,8.915105,...,1.252184,0.240235,3.078347,0.561146,8.676120,3.075108,8.197458,1.232233,4.307033,0.730423
6,CHW,AL,C,66.610743,92.661003,4.382509,172.196333,153.788645,4.036464,8.993642,...,1.360804,0.243807,2.230189,0.437781,8.590266,3.905472,8.331670,1.133913,4.736967,0.670238
7,CIN,NL,C,74.377631,95.323671,3.829257,143.660533,185.669558,5.395961,9.843240,...,1.264000,0.234111,2.887720,0.475394,9.424736,3.390052,7.982130,1.121127,4.125876,0.520000
8,CLE,AL,C,79.842712,75.646269,3.717950,190.100212,166.583989,4.402829,9.009453,...,1.250580,0.237159,3.208102,0.522854,9.140545,3.179986,8.085155,1.082312,3.998847,0.590773
9,COL,NL,W,64.268273,104.630258,4.758670,126.498119,179.689193,4.093355,8.938012,...,1.407621,0.264354,2.326963,0.410155,7.511001,3.453716,9.217012,1.293387,5.171323,0.548333


# Save Processed Data To CSV

In [779]:
train_columns=final_wins.columns
print('2026 Pitching Columns')
print(team_2026_pitching.columns)
print()
print('2026 Hitting Columns')
print(team_2026_data.columns)


team_2026 = pd.merge(team_2026_data, team_2026_pitching, on=['TEAM', 'LEAGUE', 'DIVISION'])
print()
print('Final Concatenated Columns')


team_2026['Year']=2026
team_2026['WINS']=np.nan
print(team_2026.columns)
missing_cols=['innings', 'caughtStealing', 'stolenBasesAllowed']
final_wins=final_wins[[col for col in final_wins.columns if col not in missing_cols]]
team_2026=team_2026[[col for col in final_wins.columns]]
team_2026.to_csv(os.path.join(data_dir,'Processed_2026_Team_Data.csv'),index=False)

print()
print('Final Wins (training data) Columns')
final_wins.to_csv(os.path.join(data_dir,'Processed_Historical_Team_Wins_Data.csv'),index=False)
print(final_wins.columns)
A=final_wins.columns
B=team_2026.columns
X=[col for col in A if col not in B]
Y=[col for col in B if col not in A]
if len(X)==0 and len(Y)==0:
    print('\nColumns in training and inference are a perfect match')
else:
    print('\nColumns are not perfect match')


2026 Pitching Columns
Index(['TEAM', 'LEAGUE', 'DIVISION', 'W_pitch', 'L_pitch', 'ERA_pitch',
       'G_pitch', 'GS_pitch', 'CG_pitch', 'SHO_pitch', 'SV_pitch', 'SVO_pitch',
       'IP_pitch', 'H_pitch', 'R_pitch', 'ER_pitch', 'HR_pitch', 'HB_pitch',
       'BB_pitch', 'SO_pitch', 'WHIP_pitch', 'AVG_pitch',
       'P_strikeoutWalkRatio', 'P_winPercentage', 'P_strikeoutsPer9Inn',
       'P_walksPer9Inn', 'P_hitsPer9Inn', 'P_homeRunsPer9', 'P_runsScoredPer9',
       'SV_pct'],
      dtype='object')

2026 Hitting Columns
Index(['TEAM', 'G', 'AB', 'R', 'H', '2B', '3B', 'HR', 'RBI', 'BB', 'SO', 'SB',
       'CS', 'totalBases', 'assists', 'putOuts', 'errors', 'chances',
       'doublePlays', 'triplePlays', 'throwingErrors', 'passedBall',
       'wildPitches', 'pickoffs', 'catchersInterference', 'AVG', 'OBP', 'SLG',
       'OPS', 'stolenBasePercentage', 'caughtStealingPercentage',
       'atBatsPerHomeRun', 'fielding_pct', 'rangeFactorPerGame',
       'rangeFactorPer9Inn', 'LEAGUE', 'DIVISION

## Visually Inspect One Team (Seattle Mariners)

In [781]:
print('Processed Hitting')
display(final_2026_offense[(final_2026_offense['TEAM']=='SEA')][['TEAM','Name','ProjectedOpeningDayRole', 'AVG', 'OBP', 'SLG','OPS','fielding_pct','errors',
       'stolenBasePercentage','Player_Weight','ServiceTime','Pos']])
print('Processed Pitching')
display(final_2026_pitching[(final_2026_pitching['TEAM']=='SEA')][['TEAM','Name','ProjectedOpeningDayRole', 'ERA_pitch', 'WHIP_pitch', 'SO_pitch','SV_pct',
       'P_strikeoutWalkRatio','P_Player_Weight','ServiceTime','Pos','IP_pitch']])

print('Aggregated Team Data')
display(team_2026[(team_2026['TEAM']=='SEA')][['TEAM', 'AVG', 'OBP', 'SLG','OPS','fielding_pct','errors',
       'stolenBasePercentage','ERA_pitch', 'WHIP_pitch', 'SO_pitch','SV_pct',
       'P_strikeoutWalkRatio']])

Processed Hitting


,TEAM,Name,ProjectedOpeningDayRole,AVG,OBP,SLG,OPS,fielding_pct,errors,stolenBasePercentage,Player_Weight,ServiceTime,Pos
462,SEA,Julio Rodríguez,Lineup Regular,0.274,0.331,0.469,0.800,0.989,17.0,.806,0.116330,4.000,CF
463,SEA,Randy Arozarena,Lineup Regular,0.250,0.344,0.434,0.778,0.988,13.0,.728,0.115143,5.129,LF
464,SEA,Cal Raleigh,Lineup Regular,0.226,0.314,0.484,0.798,0.994,27.0,.808,0.108021,4.085,C
465,SEA,J.P. Crawford,Lineup Regular,0.248,0.340,0.369,0.709,0.976,158.0,.700,0.108021,7.163,SS
466,SEA,Josh Naylor,Lineup Regular,0.269,0.329,0.447,0.776,0.993,52.0,.859,0.105647,6.127,1B
467,SEA,Brendan Donovan,Lineup Regular,0.282,0.361,0.411,0.772,0.987,8.0,.556,0.094964,4.000,3B
468,SEA,Dominic Canzone,Platoon vs R,0.247,0.304,0.428,0.732,0.995,2.0,.833,0.071223,2.014,DH
469,SEA,Cole Young,Platoon vs R,0.211,0.302,0.305,0.607,0.984,4.0,1.000,0.066474,0.121,2B
470,SEA,Luke Raley,Platoon vs R,0.232,0.319,0.425,0.744,0.994,6.0,.818,0.054604,3.106,RF
471,SEA,Victor Robles,Platoon vs L,0.247,0.319,0.368,0.687,0.969,2.0,.791,0.053417,7.033,OF


Processed Pitching


,TEAM,Name,ProjectedOpeningDayRole,ERA_pitch,WHIP_pitch,SO_pitch,SV_pct,P_strikeoutWalkRatio,P_Player_Weight,ServiceTime,Pos,IP_pitch
623,SEA,Bryan Woo,Starting Rotation,3.21,0.98,392,NaN,4.90,0.121043,2.121,SP,395.2
624,SEA,Luis Castillo,Starting Rotation,3.55,1.18,1493,NaN,3.30,0.111111,8.101,SP,1410.2
625,SEA,George Kirby,Starting Rotation,3.58,1.11,621,NaN,6.68,0.107387,3.151,SP,637.2
626,SEA,Logan Gilbert,Starting Rotation,3.58,1.06,884,NaN,4.88,0.101179,4.144,SP,835.1
627,SEA,Bryce Miller,Starting Rotation,4.01,1.13,364,NaN,3.47,0.087523,2.153,SP,402.0
628,SEA,Andrés Muñoz,Closer,2.43,1.04,354,0.795918,3.40,0.039727,6.080,RP,259.1
629,SEA,Jose A. Ferrer,Setup Man,4.36,1.26,121,0.631579,3.46,0.043451,2.094,RP,142.1
630,SEA,Matt Brash,Setup Man,3.31,1.38,227,0.571429,2.84,0.039106,3.121,RP,168.2
631,SEA,Eduard Bazardo,Middle Reliever,3.18,1.04,144,0.000000,3.13,0.042210,2.118,RP,141.2
632,SEA,Gabe Speier,Middle Reliever,3.64,1.11,214,0.111111,4.20,0.036002,4.000,RP,180.1


Aggregated Team Data


,TEAM,AVG,OBP,SLG,OPS,fielding_pct,errors,stolenBasePercentage,ERA_pitch,WHIP_pitch,SO_pitch,SV_pct,P_strikeoutWalkRatio
23,SEA,0.249925,0.329331,0.414666,0.743997,0.987502,102.092353,0.780592,3.81152,1.179264,1151.617325,0.737594,3.894826
